# Notebook 06 · Qualify the offline evaluator and freeze fresh tasks

Fresh-run release. Follow the input and review gates in RUN_ORDER.md.

In [1]:
import csv
import hashlib
import io
import json
import os
import re
import zipfile
from pathlib import Path, PurePosixPath

INPUT = Path(os.environ.get("OPENAPI_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("OPENAPI_WORK", "/kaggle/working"))
DESIGN = "openapi-fresh-pilot-v1"


def sha(data):
    return hashlib.sha256(data).hexdigest()


def file_sha(path):
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()


def encode(data):
    return (json.dumps(data, sort_keys=True, ensure_ascii=False, separators=(",", ":"),
                       allow_nan=False) + "\n").encode()


def load(path):
    return json.loads(path.read_bytes())


def records(path):
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


def csv_records(path):
    with path.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


def atomic(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_bytes(data)
    tmp.replace(path)


def jsonl(path, rows):
    atomic(path, b"".join(map(encode, rows)))


def csv_write(path, fields, rows):
    output = io.StringIO(newline="")
    writer = csv.DictWriter(output, fieldnames=fields, lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)
    atomic(path, output.getvalue().encode())


def safe_path(name):
    p = PurePosixPath(name)
    if p.is_absolute() or ".." in p.parts or "\\" in name or p.as_posix() != name:
        raise ValueError(f"Unsafe manifest path: {name}")
    return Path(*p.parts)


import hashlib
import json
import re
from pathlib import PurePosixPath


ARTIFACT_SCHEMA = "openapi-artifact-v1"
PRODUCER_NOTEBOOKS = {
    "00": "00_protocol_and_runtime",
    "01": "01_corpus_preparation_and_eligibility",
    "02": "02_operation_slicing_and_prose_measurement",
    "03": "03_pair_and_evaluator_feasibility",
    "04": "04_strata_and_pilot_selection",
    "05": "05_oracle_build_and_review",
    "06": "06_build_corrected_evaluator_and_generation_tasks",
    "07": "07_pilot_generation",
    "08": "08_pilot_execution_and_review",
}


def provenance_digest(value):
    data = json.dumps(value, sort_keys=True, separators=(",", ":"),
                      ensure_ascii=False, allow_nan=False).encode("utf-8")
    return hashlib.sha256(data).hexdigest()


def artifact_path_for(record):
    paths = {record[key] for key in ("artifact_path", "relative_path", "path")
             if key in record}
    if len(paths) != 1:
        raise RuntimeError("Missing or inconsistent artifact paths")
    name = paths.pop()
    path = PurePosixPath(name)
    if not name or name == "." or path.is_absolute() or ".." in path.parts or "\\" in name or path.as_posix() != name:
        raise RuntimeError(f"Invalid artifact path: {name!r}")
    return name


def manifest_digest_valid(value):
    return isinstance(value, str) and re.fullmatch(r"[0-9a-f]{64}", value) is not None


def register_artifact_provenance(manifest):
    stage = manifest["stage"]
    artifacts = manifest["artifacts"]
    if "upstream_manifest_sha256" not in manifest:
        if stage != "00":
            raise RuntimeError(f"Stage {stage} has no upstream identity")
        manifest["upstream_manifest_sha256"] = {}
    links = manifest["upstream_manifest_sha256"]
    config_hash = manifest.get("configuration_hash")
    if config_hash is None:
        config_hash = provenance_digest({artifact_path_for(row): row["sha256"]
                                         for row in artifacts
                                         if artifact_path_for(row).startswith("config/")})
    manifest["configuration_hash"] = config_hash
    manifest["artifact_record_schema"] = ARTIFACT_SCHEMA
    upstream_hash = provenance_digest(links)
    for row in artifacts:
        row.update(artifact_path=artifact_path_for(row), schema_version=ARTIFACT_SCHEMA,
                   producer_notebook=PRODUCER_NOTEBOOKS[stage], config_hash=config_hash,
                   upstream_manifest_hash=upstream_hash)
    verify_artifact_provenance(manifest)


def verify_artifact_provenance(manifest):
    stage = manifest["stage"]
    if manifest.get("artifact_record_schema") != ARTIFACT_SCHEMA:
        raise RuntimeError(f"Stage {stage} requires a corrected provenance export")
    links = manifest["upstream_manifest_sha256"]
    config_hash = manifest["configuration_hash"]
    if (not isinstance(links, dict) or bool(links) != (stage != "00")
            or any(key not in PRODUCER_NOTEBOOKS or key >= stage or not manifest_digest_valid(value)
                   for key, value in links.items()) or not manifest_digest_valid(config_hash)):
        raise RuntimeError(f"Invalid Stage {stage} provenance")
    expected = (ARTIFACT_SCHEMA, PRODUCER_NOTEBOOKS[stage], config_hash, provenance_digest(links))
    seen = set()
    artifacts = manifest["artifacts"]
    if not artifacts or len(artifacts) != manifest["artifact_count"]:
        raise RuntimeError(f"Invalid Stage {stage} artifact count")
    for row in artifacts:
        name = artifact_path_for(row)
        actual = tuple(row.get(key) for key in
                       ("schema_version", "producer_notebook", "config_hash", "upstream_manifest_hash"))
        if (row.get("artifact_path") != name or name in seen or actual != expected
                or not manifest_digest_valid(row.get("sha256"))
                or type(row.get("bytes")) is not int or row["bytes"] < 0):
            raise RuntimeError(f"Invalid Stage {stage} artifact provenance: {name}")
        seen.add(name)


def stage_input(stage, *, fresh=True):
    matches = []
    for external in sorted(INPUT.rglob(f"stage_{stage}_manifest.json")):
        sidecar = load(external)
        if fresh and sidecar.get("design_id") != DESIGN:
            continue
        for manifest in external.parent.rglob(f"registries/{stage}_manifest.json"):
            matches.append((external, manifest))
    if len(matches) != 1:
        raise RuntimeError(f"Attach one expanded Stage {stage} release. Found {len(matches)}.")
    external, internal = matches[0]
    sidecar, manifest = load(external), load(internal)
    expected = sidecar.get("internal_manifest_sha256") or sidecar.get("internal_manifest", {}).get("sha256")
    if expected != file_sha(internal):
        raise RuntimeError(f"Stage {stage} manifest digest mismatch.")
    if (manifest.get("stage_status", manifest.get("status")) != "PASS"
            or sidecar.get("stage_status", sidecar.get("status")) != "PASS"):
        raise RuntimeError(f"Stage {stage} has no passing handoff.")
    verify_artifact_provenance(manifest)
    root = internal.parent.parent
    paths = {}
    for item in manifest["artifacts"]:
        name = item.get("relative_path") or item.get("artifact_path") or item.get("path")
        path = root / safe_path(name)
        if name in paths or not path.is_file() or path.stat().st_size != item["bytes"] or file_sha(path) != item["sha256"]:
            raise RuntimeError(f"Stage {stage} artifact mismatch: {name}")
        paths[name] = item["sha256"]
    if len(paths) != manifest["artifact_count"]:
        raise RuntimeError(f"Stage {stage} artifact count mismatch.")
    return root, manifest, expected, paths


def pack(stage, root, metadata):
    output = WORK / f"stage_{stage}_fresh_dataset"
    output.mkdir(parents=True, exist_ok=True)
    manifest_path = root / f"registries/{stage}_manifest.json"
    files = sorted(p for p in root.rglob("*") if p.is_file() and p != manifest_path
                   and "__pycache__" not in p.parts and not p.name.endswith(".tmp"))
    artifacts = [{"relative_path": p.relative_to(root).as_posix(), "bytes": p.stat().st_size,
                  "sha256": file_sha(p)} for p in files]
    manifest = {"stage": stage, "status": "PASS", "design_id": DESIGN,
                "artifact_count": len(artifacts), "artifacts": artifacts, **metadata}
    register_artifact_provenance(manifest)
    atomic(manifest_path, encode(manifest))
    archive = output / f"stage_{stage}_fresh.zip"
    tmp = archive.with_suffix(".tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, compresslevel=4) as z:
        for p in [*files, manifest_path]:
            z.write(p, "prose_operation_experiment/" + p.relative_to(root).as_posix())
    tmp.replace(archive)
    external = {k: v for k, v in manifest.items() if k != "artifacts"}
    external.update(archive=archive.name, archive_sha256=file_sha(archive),
                    archive_bytes=archive.stat().st_size, internal_manifest_sha256=file_sha(manifest_path))
    atomic(output / f"stage_{stage}_manifest.json", encode(external))
    atomic(output / f"stage_{stage}_checksums.sha256",
           f'{external["archive_sha256"]}  {archive.name}\n'.encode())
    atomic(output / "README.md", f"# Stage {stage} fresh pilot\n\nDesign {DESIGN}.\n".encode())
    print(f"Stage {stage} PASS. Publish {output}")
    return external

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def sha256_file(path):

    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(
            f"Cannot hash a missing file or directory: {path}"
        )

    hasher = hashlib.sha256()

    with path.open("rb") as stream:
        for chunk in iter(
            lambda: stream.read(4 * 1024 * 1024),
            b"",
        ):
            hasher.update(chunk)

    return hasher.hexdigest()


digest = sha256_bytes
file_hash = sha256_file


def json_bytes(value):

    return (
        json.dumps(
            value,
            sort_keys=True,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n"
    ).encode("utf-8")


def jsonl_bytes(rows):

    return b"".join(
        (
            json.dumps(
                row,
                sort_keys=True,
                ensure_ascii=False,
                separators=(",", ":"),
                allow_nan=False,
            )
            + "\n"
        ).encode("utf-8")
        for row in rows
    )


def read_json(value):

    if isinstance(value, (str, Path)):
        return json.loads(
            Path(value).read_text(encoding="utf-8")
        )

    return json.loads(value)


def read_jsonl(value):

    if isinstance(value, Path):
        value = value.read_bytes()

    elif isinstance(value, str):
        if "\n" not in value and Path(value).is_file():
            value = Path(value).read_bytes()

    if isinstance(value, bytes):
        value = value.decode("utf-8")

    return [
        json.loads(line)
        for line in value.splitlines()
        if line.strip()
    ]


# Stable compatibility names used by later cells.
canonical_json_bytes = json_bytes
canonical_jsonl_bytes = jsonl_bytes
read_jsonl_bytes = read_jsonl
read_jsonl_file = read_jsonl


# ============================================================
# 3. Immutable artifact writing
# ============================================================

def write_immutable(path, content):

    path = Path(path)

    if isinstance(content, str):
        content = content.encode("utf-8")

    if not isinstance(content, bytes):
        raise TypeError(
            "Expected bytes or UTF-8 text"
        )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if path.exists():

        if (
            not path.is_file()
            or path.read_bytes() != content
        ):
            raise RuntimeError(
                f"Changed immutable artifact: {path}"
            )

        return "REUSED_IDENTICAL"

    with tempfile.NamedTemporaryFile(
        dir=path.parent,
        delete=False,
    ) as output:

        temporary_path = Path(output.name)

        output.write(content)
        output.flush()
        os.fsync(output.fileno())

    try:
        os.replace(
            temporary_path,
            path,
        )

    finally:
        temporary_path.unlink(
            missing_ok=True
        )

    return "WRITTEN"


write_once = write_immutable


def write_json(path, value):

    return write_immutable(
        path,
        json_bytes(value),
    )


write_json_immutable = write_json


# ============================================================
# 4. Notebook checkpoint utilities
# ============================================================

def save_cell_report(number, report=None, **values):

    report = dict(report or {})

    report.update(values)

    report.setdefault("status", "PASS")
    report.setdefault("notebook", "06")
    report.setdefault("cell", int(number))

    report_path = (
        ROOT
        / "reports"
        / "audit"
        / f"cell{int(number):02d}_stage06.json"
    )

    write_json(
        report_path,
        report,
    )

    return report


def require_cell_pass(number):

    report_path = (
        ROOT
        / "reports"
        / "audit"
        / f"cell{int(number):02d}_stage06.json"
    )

    report = read_json(report_path)

    if report.get("status") != "PASS":
        raise RuntimeError(
            f"Notebook 06 Cell {number} has not passed"
        )

    return report


require_checkpoint = require_cell_pass


def checkpoint(number, **values):

    return save_cell_report(
        number,
        **values,
    )



import sys
import tempfile
import time
from collections import Counter
from pathlib import PurePosixPath

stage06_started = time.time()
ROOT = PROJECT_ROOT = WORK / "fresh_release/stage_06_single_replay/prose_operation_experiment"
STAGE_06_DATASET_ROOT = WORK / "stage_06_fresh_dataset"
STAGE_06_DATASET_ROOT.mkdir(parents=True, exist_ok=True)
STAGES = ("00", "04", "05")
for directory in ("config", "src", "data/inventory", "reports/audit"):
    (PROJECT_ROOT / directory).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# The verified manifest digests below are carried forward by each stage.
STAGE_INPUTS = {stage: stage_input(stage, fresh=stage != "00") for stage in STAGES}
root00, manifest00, hash00, paths00 = STAGE_INPUTS["00"]
root04, manifest04, hash04, paths04 = STAGE_INPUTS["04"]
root05, manifest05, hash05, paths05 = STAGE_INPUTS["05"]
if (manifest04["upstream_manifest_sha256"]["00"] != hash00
        or manifest05["upstream_manifest_sha256"]["00"] != hash00
        or manifest05["upstream_manifest_sha256"]["04"] != hash04
        or manifest05["upstream_manifest_sha256"]["03"] != manifest04["upstream_manifest_sha256"]["03"]):
    raise RuntimeError("Stage 00, 04 and 05 are from different experimental runs.")
if not all(manifest05.get(key) is True for key in (
        "checkpoint_b_reached", "may_begin_notebook_06", "human_review_complete")):
    raise RuntimeError("Checkpoint B requires approved human review of every oracle.")
if manifest05.get("model_generation_performed") is not False:
    raise RuntimeError("Stage 05 must precede model generation.")


class VerifiedStage:
    def __init__(self, stage, input_data):
        self.root, self.manifest, self.internal_manifest_sha256, paths = input_data
        self.members = set(paths) | {f"registries/{stage}_manifest.json"}

    def read(self, name):
        if name not in self.members:
            raise RuntimeError("Artifact is absent from the verified manifest: " + name)
        return (self.root / safe_path(name)).read_bytes()


STAGE_ARCHIVES = {stage: VerifiedStage(stage, stage_data) for stage, stage_data in STAGE_INPUTS.items()}
selection = list(records(root05 / "partitions/pilot/final_selected_operations.jsonl"))
if (len(selection) != 100 or len({r["op_id"] for r in selection}) != 100
        or len({r["pair_id"] for r in selection}) != 100
        or dict(Counter(r["stratum"] for r in selection)) != manifest05["stratum_counts"]
        or any(not r["family_id"] or not 0 < r["inclusion_probability"] <= 1 for r in selection)):
    raise RuntimeError("Invalid Stage 05 sample or inclusion probabilities.")
strata = {row["stratum"]: row for row in records(root05 / "partitions/pilot/final_stratum_sizes.jsonl")}
for row in selection:
    h = strata[row["stratum"]]
    if (row["inclusion_probability"] != h["n_h"] / h["N_h"]
            or row["stratum_population_share"] != h["W_h"]):
        raise RuntimeError("Stage 05 selection weights disagree with the final stratum frame.")
write_immutable(PROJECT_ROOT / "data/inventory/final_selected_operations.json", json_bytes(selection))
save_cell_report(1, checkpoint_b_reached=True, operation_count=len(selection),
                 upstream_manifest_sha256={stage: data[2] for stage, data in STAGE_INPUTS.items()})
print("Stage 06 verified Checkpoint B and the fresh Stage 00/04/05 chain.")


Stage 06 verified Checkpoint B and the fresh Stage 00/04/05 chain.


In [2]:
require_cell_pass(1)
verification = {stage: {"status": "PASS", "stage": stage, "artifact_count": len(data[3]),
                        "manifest_sha256": data[2]} for stage, data in STAGE_INPUTS.items()}
for stage, report in verification.items():
    write_json_immutable(PROJECT_ROOT / f"reports/audit/upstream_stage_{stage}_verified.json", report)
write_json_immutable(PROJECT_ROOT / "reports/audit/upstream_stage_00_04_05_verification.json", verification)
save_cell_report(2, verified_stages=list(STAGES), human_review_complete=True)
print("Stage 06 verified all upstream registered bytes.")


Stage 06 verified all upstream registered bytes.


In [3]:
# NOTEBOOK 06 — CELL 3
# Qualify the CPU runtime and freeze required dependencies.

from __future__ import annotations

from importlib.metadata import (
    PackageNotFoundError,
    version,
)
from pathlib import Path

import importlib
import json
import os
import platform
import shutil
import sys
import time


CELL3_STARTED = time.time()

require_cell_pass(2)

REQUIRED_DEPENDENCIES = {
    "pytest": "pytest",
    "requests": "requests",
    "jsonschema": "jsonschema",
    "yaml": "PyYAML",
    "numpy": "numpy",
    "scipy": "scipy",
    "statsmodels": "statsmodels",
    "transformers": "transformers",
    "tokenizers": "tokenizers",
}

dependency_versions = {}
missing_dependencies = []

for import_name, distribution in (
    REQUIRED_DEPENDENCIES.items()
):
    try:
        importlib.import_module(
            import_name
        )

        dependency_versions[distribution] = (
            version(distribution)
        )

    except (
        ImportError,
        PackageNotFoundError,
    ):
        missing_dependencies.append(
            distribution
        )

if missing_dependencies:
    raise RuntimeError(
        "Missing Notebook 06 dependencies:\n"
        + "\n".join(
            f"- {name}"
            for name in missing_dependencies
        )
    )

if sys.version_info < (3, 10):
    raise RuntimeError(
        "Notebook 06 requires Python 3.10 or newer."
    )

if platform.system() != "Linux":
    raise RuntimeError(
        "Linux runtime required for sandbox qualification."
    )

for directory in (
    "config",
    "src",
    "data/inputs/L0",
    "data/inputs/L1",
    "data/prompts/pilot",
    "data/audits",
    "data/inventory",
    "reports/audit",
    "reports/qualification",
    "registries",
):
    (
        PROJECT_ROOT / directory
    ).mkdir(
        parents=True,
        exist_ok=True,
    )

stage00 = STAGE_ARCHIVES["00"]

for relative_path in (
    "config/system_prompt.txt",
    "config/user_prompt.txt",
    "config/runtime.yaml",
    "config/study.yaml",
):
    write_immutable(
        PROJECT_ROOT / relative_path,
        stage00.read(relative_path),
    )

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "cpu_count": os.cpu_count(),
    "packages": dependency_versions,
    "chroot_binary": shutil.which("chroot"),
    "worker_isolation": "chroot_drop_uid_gid_seccomp",
    "user_namespaces_required": False,
    "execution_mode": "CPU_ONLY",
}

write_json_immutable(
    PROJECT_ROOT
    / "config/stage06_environment.json",
    environment,
)

cell3_report = save_cell_report(
    3,
    dependency_count=len(dependency_versions),
    python_version=sys.version.split()[0],
    os=platform.system(),
    sandbox_binary_detected=bool(
        shutil.which("chroot")
    ),
)

print(
    json.dumps(
        cell3_report,
        indent=2,
    )
)

print("\nNOTEBOOK 06 — CELL 3: PASS")

{
  "dependency_count": 9,
  "python_version": "3.13.18",
  "os": "Linux",
  "sandbox_binary_detected": true,
  "status": "PASS",
  "notebook": "06",
  "cell": 3
}

NOTEBOOK 06 — CELL 3: PASS


In [4]:
from collections import Counter

require_cell_pass(3)
reconstruction_ledger = []
for row in selection:
    op = row["op_id"]
    output = {"op_id": op, "pair_id": row["pair_id"], "stratum": row["stratum"],
              "selection_status": row["final_selection_status"]}
    for condition in ("L0", "L1"):
        source = f"partitions/pilot/inputs/{condition}/{op}.json"
        raw = STAGE_ARCHIVES["05"].read(source)
        digest = sha256_bytes(raw)
        if digest != row[f"{condition}_hash"]:
            raise RuntimeError("Pair hash mismatch: " + source)
        json.loads(raw)
        write_immutable(PROJECT_ROOT / f"data/inputs/{condition}/{op}.json", raw)
        output[f"{condition}_sha256"] = digest
        output[f"{condition}_source_stage"] = "05"
    reconstruction_ledger.append(output)
write_immutable(PROJECT_ROOT / "data/audits/cell04_pair_reconstruction.jsonl",
                canonical_jsonl_bytes(reconstruction_ledger))
save_cell_report(4, operation_count=len(reconstruction_ledger),
                 selection_counts=dict(Counter(r["selection_status"] for r in reconstruction_ledger)))
print("Stage 06 reconstructed 100 matched pairs from the human-reviewed Stage 05 release.")


Stage 06 reconstructed 100 matched pairs from the human-reviewed Stage 05 release.


In [5]:
# NOTEBOOK 06 — CELL 5
# Verify all frozen oracle records, record hashes,
# and the nonzero primary metric denominators.

from __future__ import annotations

from pathlib import Path

import json
import jsonschema
import time


CELL5_STARTED = time.time()

require_cell_pass(4)

stage05 = STAGE_ARCHIVES["05"]

selection = read_json(
    PROJECT_ROOT
    / "data/inventory/final_selected_operations.json"
)

contract_bytes = stage05.read(
    "config/oracle_record_contracts.json"
)

contracts = json.loads(
    contract_bytes
)["records"]

write_immutable(
    PROJECT_ROOT
    / "config/oracle_record_contracts.json",
    contract_bytes,
)

ORACLE_RECORD_TYPES = (
    "requirements",
    "scenarios",
    "baselines",
    "legitimate_variations",
    "response_faults",
    "interface_items",
)

denominator_ledger = []

for index, selected in enumerate(
    selection,
    start=1,
):
    op_id = selected["op_id"]

    prefix = (
        f"oracles/frozen/pilot/{op_id}/"
    )

    validation_bytes = stage05.read(
        prefix + "validation.json"
    )

    validation = json.loads(
        validation_bytes
    )

    if (
        validation.get("status") != "PASS"
        or validation.get("op_id") != op_id
        or validation.get("human_review_complete") is not True
    ):
        raise RuntimeError(
            "Unqualified oracle package: "
            + op_id
        )

    records_by_type = {}

    for record_type in ORACLE_RECORD_TYPES:
        filename = (
            record_type + ".jsonl"
        )

        raw = stage05.read(
            prefix + filename
        )

        expected_hash = validation[
            "artifact_sha256"
        ][filename]

        if sha256_bytes(raw) != expected_hash:
            raise RuntimeError(
                f"Oracle hash mismatch: "
                f"{op_id}/{filename}"
            )

        records = read_jsonl_bytes(
            raw
        )

        if record_type in contracts:
            validator = (
                jsonschema.Draft202012Validator(
                    contracts[record_type]
                )
            )

            for row_number, record in enumerate(
                records,
                start=1,
            ):
                error = next(
                    validator.iter_errors(record),
                    None,
                )

                if error is not None:
                    raise RuntimeError(
                        f"Oracle contract failure: "
                        f"{op_id}/{filename}/"
                        f"{row_number}: {error.message}"
                    )

        write_immutable(
            PROJECT_ROOT
            / prefix
            / filename,
            raw,
        )

        records_by_type[
            record_type
        ] = records

    supplements_path = (
        prefix
        + "response_fault_supplements.jsonl"
    )

    if supplements_path in stage05.members:
        supplement_bytes = stage05.read(
            supplements_path
        )

        write_immutable(
            PROJECT_ROOT
            / supplements_path,
            supplement_bytes,
        )

        records_by_type[
            "response_faults"
        ].extend(
            read_jsonl_bytes(
                supplement_bytes
            )
        )

    write_immutable(
        PROJECT_ROOT
        / prefix
        / "validation.json",
        validation_bytes,
    )

    applicable_requirements = {
        record["requirement_id"]
        for record in records_by_type[
            "requirements"
        ]
        if record.get("applicability")
        == "APPLICABLE"
    }

    applicable_scenarios = {
        record["scenario_id"]
        for record in records_by_type[
            "scenarios"
        ]
        if record.get("applicability")
        == "APPLICABLE"
    }

    applicable_faults = {
        record["fault_id"]
        for record in records_by_type[
            "response_faults"
        ]
        if (
            record.get(
                "applicable",
                True,
            )
            and record.get(
                "applicability",
                "APPLICABLE",
            ) == "APPLICABLE"
        )
    }

    baseline_count = len(
        records_by_type[
            "baselines"
        ]
    )

    if not (
        applicable_requirements
        and applicable_scenarios
        and applicable_faults
        and baseline_count > 0
    ):
        raise RuntimeError(
            "Zero primary oracle denominator: "
            + op_id
        )

    fault_record_count = len(
        records_by_type[
            "response_faults"
        ]
    )

    if len(applicable_faults) != fault_record_count:
        raise RuntimeError(
            "Duplicate or inapplicable frozen fault: "
            + op_id
        )

    denominator_ledger.append({
        "op_id": op_id,
        "pair_id": selected["pair_id"],
        "stratum": selected["stratum"],
        "requirements": len(
            applicable_requirements
        ),
        "scenarios": len(
            applicable_scenarios
        ),
        "baselines": baseline_count,
        "faults": len(
            applicable_faults
        ),
        "variations": len(
            records_by_type[
                "legitimate_variations"
            ]
        ),
    })

    if index % 20 == 0:
        print({
            "oracles_audited": index,
            "oracle_total": 100,
        })

if len(denominator_ledger) != 100:
    raise RuntimeError(
        "Expected exactly 100 qualified oracles."
    )

ledger_path = (
    PROJECT_ROOT
    / "data/audits/"
    "cell05_frozen_denominators.jsonl"
)

write_immutable(
    ledger_path,
    canonical_jsonl_bytes(
        denominator_ledger
    ),
)

total_faults = sum(
    row["faults"]
    for row in denominator_ledger
)

if total_faults < len(selection):
    raise RuntimeError(
        f"Unexpected frozen fault count: {total_faults}"
    )

cell5_report = save_cell_report(
    5,
    oracle_package_count=100,
    total_applicable_fault_count=total_faults,
    zero_primary_denominator_count=0,
    denominator_ledger_sha256=sha256_file(
        ledger_path
    ),
)

print(
    json.dumps(
        cell5_report,
        indent=2,
    )
)

print("\nNOTEBOOK 06 — CELL 5: PASS")

{'oracles_audited': 20, 'oracle_total': 100}
{'oracles_audited': 40, 'oracle_total': 100}
{'oracles_audited': 60, 'oracle_total': 100}
{'oracles_audited': 80, 'oracle_total': 100}
{'oracles_audited': 100, 'oracle_total': 100}
{
  "oracle_package_count": 100,
  "total_applicable_fault_count": 733,
  "zero_primary_denominator_count": 0,
  "denominator_ledger_sha256": "d5e6b335530968a32d55c20e94a020762b32500578ea2d6ebc41a7b6ee71baf0",
  "status": "PASS",
  "notebook": "06",
  "cell": 5
}

NOTEBOOK 06 — CELL 5: PASS


In [6]:
# NOTEBOOK 06 — CELL 6
# Freeze seven primary endpoints and associated policies.

from __future__ import annotations

from pathlib import Path

import json
import time
import yaml


CELL6_STARTED = time.time()

require_cell_pass(5)

PRIMARY_ENDPOINT_DEFINITIONS = [
    {
        "name": "collection_success",
        "unit": "binary",
        "numerator": (
            "One if an eligible module collects "
            "at least one pytest item without errors"
        ),
        "denominator": "One expected generated artifact",
        "zero_rule": "Not applicable; denominator fixed at one",
        "failure_rule": (
            "Zero for missing, malformed, unsafe, "
            "compile-failed or collection-failed output"
        ),
        "estimand_type": "full_population",
    },
    {
        "name": "baseline_pass",
        "unit": "binary",
        "numerator": (
            "One if at least one item passes normally "
            "and no item fails, errors or unexpectedly passes"
        ),
        "denominator": "One expected generated artifact",
        "zero_rule": "Not applicable; denominator fixed at one",
        "failure_rule": (
            "Zero for invalid baseline, all-skipped, "
            "xfail-only or failed execution"
        ),
        "estimand_type": "full_population",
    },
    {
        "name": "request_intent_match",
        "unit": "fraction",
        "numerator": (
            "Distinct classifiable request attempts matching "
            "a frozen positive or negative scenario"
        ),
        "denominator": "Distinct classifiable request attempts",
        "zero_rule": (
            "Unavailable when no requests are classifiable"
        ),
        "failure_rule": (
            "Unclassifiable requests are reported separately; "
            "no invented matching credit"
        ),
        "estimand_type": "common_evaluable_pairs",
    },
    {
        "name": "selected_scenario_coverage",
        "unit": "fraction",
        "numerator": (
            "Distinct applicable frozen scenarios exercised "
            "by observed requests"
        ),
        "denominator": "All applicable frozen scenarios",
        "zero_rule": (
            "Zero applicable scenarios blocks operation eligibility"
        ),
        "failure_rule": (
            "Zero when generated output exercises no scenario"
        ),
        "estimand_type": "full_population",
    },
    {
        "name": "supported_assertion_score",
        "unit": "fraction",
        "numerator": "Reviewed assertion claims judged supported",
        "denominator": "All usable reviewed assertion claims",
        "zero_rule": (
            "Full-population zero credit when no claims exist; "
            "literal observed-claim fraction remains unavailable"
        ),
        "failure_rule": (
            "Unsupported or unreached claims receive no credit"
        ),
        "estimand_type": "full_population",
    },
    {
        "name": "effective_requirement_check_coverage",
        "unit": "fraction",
        "numerator": (
            "Distinct requirements with reached, grounded, "
            "qualified assertion-linked evidence"
        ),
        "denominator": "All applicable frozen requirements",
        "zero_rule": (
            "Zero applicable requirements blocks eligibility"
        ),
        "failure_rule": (
            "No credit for mere requests, schema imports "
            "or requirement mentions"
        ),
        "estimand_type": "full_population",
    },
    {
        "name": "unconditional_response_fault_detection",
        "unit": "fraction",
        "numerator": (
            "Distinct applicable frozen response faults "
            "consumed and killed"
        ),
        "denominator": "All applicable frozen response faults",
        "zero_rule": (
            "Zero applicable faults blocks eligibility"
        ),
        "failure_rule": (
            "Baseline-invalid and not-consumed faults receive "
            "no kills; full denominator remains unchanged"
        ),
        "estimand_type": "full_population",
    },
]

for endpoint in PRIMARY_ENDPOINT_DEFINITIONS:
    endpoint["status"] = "primary"
    endpoint["eligible_evidence"] = (
        "Frozen Stage 05 oracles and reconciled "
        "Stage 08 execution/reviewer evidence"
    )

METRICS_CONTRACT = {
    "schema_version": "stage06-seven-primary-metrics-v1",
    "partition": "pilot",
    "primary_endpoint_count": 7,
    "paired_direction": "L1_minus_L0",
    "primary_endpoints": PRIMARY_ENDPOINT_DEFINITIONS,
    "pytest_item_policy": {
        "count_collected_items_not_functions": True,
        "all_skipped_baseline_pass": 0,
        "xfail_only_baseline_pass": 0,
        "vacuous_pass_reported_separately": True,
        "unexpected_pass_is_failure": True,
    },
    "evaluator_failure_policy": (
        "Evaluator bugs or environment failures are "
        "separate blockers; never recode as model failures"
    ),
    "unresolved_evaluator_limitation_policy": {
        "affected_endpoints": "retain task, report unavailable with reason",
        "full_population_inference": "stop until missing evidence policy is resolved",
    },
    "secondary_diagnostics": [
        "literal_pooled_supported_claim_fraction",
        "conditional_response_fault_detection",
        "legitimate_variation_acceptance",
        "unknown_request_intent",
        "vacuous_baseline_pass",
        "pytest_skip_xfail_counts",
    ],
}

metrics_path = (
    PROJECT_ROOT
    / "config/metrics.yaml"
)

metrics_bytes = yaml.safe_dump(
    METRICS_CONTRACT,
    sort_keys=False,
    allow_unicode=True,
).encode("utf-8")

write_immutable(
    metrics_path,
    metrics_bytes,
)

if len(PRIMARY_ENDPOINT_DEFINITIONS) != 7:
    raise RuntimeError(
        "Exactly seven primary endpoints required."
    )

cell6_report = save_cell_report(
    6,
    primary_endpoint_count=7,
    metrics_contract_sha256=sha256_file(
        metrics_path
    ),
    paired_direction="L1_minus_L0",
)

print(
    json.dumps(
        cell6_report,
        indent=2,
    )
)

print("\nNOTEBOOK 06 — CELL 6: PASS")

{
  "primary_endpoint_count": 7,
  "metrics_contract_sha256": "6d8ccd8a16e3ca2fe0f9c2fdeb30b9c18df94358856b3806d035e7e99241fe4a",
  "paired_direction": "L1_minus_L0",
  "status": "PASS",
  "notebook": "06",
  "cell": 6
}

NOTEBOOK 06 — CELL 6: PASS


In [7]:
# NOTEBOOK 06 — CELL 7
# Implement the seven primary metric computations.
# Source is visible Python, not an escaped module string.

# BEGIN_STAGE06_METRICS_MODULE

from __future__ import annotations


PRIMARY_ENDPOINTS = (
    "collection_success",
    "baseline_pass",
    "request_intent_match",
    "selected_scenario_coverage",
    "supported_assertion_score",
    "effective_requirement_check_coverage",
    "unconditional_response_fault_detection",
)


def metric_record(
    endpoint,
    numerator,
    denominator,
    failure_reason=None,
    zero_credit=False,
):
    observed = (
        denominator > 0
        or zero_credit
    )

    if denominator > 0:
        score = numerator / denominator
    elif zero_credit:
        score = 0.0
    else:
        score = None

    return {
        "endpoint": endpoint,
        "numerator": numerator,
        "denominator": denominator,
        "score": score,
        "observed": observed,
        "literal_denominator_observed": (
            denominator > 0
        ),
        "failure_reason": failure_reason,
        "estimand_type": (
            "common_evaluable_pairs"
            if endpoint == "request_intent_match"
            else "full_population"
        ),
    }


def score_operation_condition(
    evidence,
    oracle,
):
    """
    Calculate scores using execution evidence and frozen oracles.

    The evaluator must not inspect the generated task's
    L0/L1 condition when judging correctness.
    """

    if evidence.get("evaluator_error"):
        raise RuntimeError(
            "Evaluator failure cannot be scored "
            "as a generated-model failure."
        )

    scenario_ids = {
        row["scenario_id"]
        for row in oracle["scenarios"]
        if row.get("applicability")
        == "APPLICABLE"
    }

    requirement_ids = {
        row["requirement_id"]
        for row in oracle["requirements"]
        if row.get("applicability")
        == "APPLICABLE"
    }

    fault_ids = {
        row["fault_id"]
        for row in oracle["faults"]
        if (
            row.get("applicable", True)
            and row.get(
                "applicability",
                "APPLICABLE",
            ) == "APPLICABLE"
        )
    }

    if not (
        scenario_ids
        and requirement_ids
        and fault_ids
        and oracle.get("baselines")
    ):
        raise ValueError(
            "Zero frozen oracle denominator."
        )

    collection_success = (
        evidence.get("collection_status")
        == "PASS"
        and int(
            evidence.get("collected_items")
            or 0
        ) > 0
    )

    item_outcomes = evidence.get(
        "item_outcomes",
        [],
    )

    has_normal_pass = (
        "passed" in item_outcomes
    )

    no_unexpected_outcomes = (
        bool(item_outcomes)
        and all(
            outcome in (
                "passed",
                "skipped",
                "xfailed",
            )
            for outcome in item_outcomes
        )
    )

    baseline_pass = (
        collection_success
        and evidence.get("baseline_status")
        == "COMPLETE"
        and has_normal_pass
        and no_unexpected_outcomes
    )

    requests = evidence.get(
        "request_attempts",
        [],
    )

    distinct_attempts = {
        row["attempt_id"]: row
        for row in requests
        if row.get("attempt_id")
    }

    classifiable = [
        row
        for row in distinct_attempts.values()
        if row.get("classifiable") is True
    ]

    matching = [
        row
        for row in classifiable
        if row.get("matched_intent") is True
    ]

    exercised_scenarios = {
        row["scenario_id"]
        for row in distinct_attempts.values()
        if row.get("scenario_id")
        in scenario_ids
    }

    reviewed_claims = [
        row
        for row in evidence.get(
            "reviewed_claims",
            [],
        )
        if row.get("reviewed") is True
    ]

    supported_claims = [
        row
        for row in reviewed_claims
        if row.get("supported") is True
    ]

    checked_requirements = {
        row["requirement_id"]
        for row in evidence.get(
            "effective_checks",
            [],
        )
        if (
            row.get("requirement_id")
            in requirement_ids
            and row.get("reached") is True
            and row.get("grounded") is True
            and row.get("linkage_qualified")
            is True
        )
    }

    killed_faults = {
        row["fault_id"]
        for row in evidence.get(
            "mutation_results",
            [],
        )
        if (
            row.get("fault_id") in fault_ids
            and row.get("result") == "KILLED"
            and row.get("consumed") is True
            and baseline_pass
        )
    }

    results = [
        metric_record(
            "collection_success",
            int(collection_success),
            1,
            evidence.get(
                "collection_failure_reason"
            ),
        ),
        metric_record(
            "baseline_pass",
            int(baseline_pass),
            1,
            evidence.get(
                "baseline_failure_reason"
            ),
        ),
        metric_record(
            "request_intent_match",
            len(matching),
            len(classifiable),
            (
                "NO_CLASSIFIABLE_REQUESTS"
                if not classifiable
                else None
            ),
        ),
        metric_record(
            "selected_scenario_coverage",
            len(exercised_scenarios),
            len(scenario_ids),
        ),
        metric_record(
            "supported_assertion_score",
            len(supported_claims),
            len(reviewed_claims),
            (
                "NO_USABLE_CLAIMS"
                if not reviewed_claims
                else None
            ),
            zero_credit=True,
        ),
        metric_record(
            "effective_requirement_check_coverage",
            len(checked_requirements),
            len(requirement_ids),
        ),
        metric_record(
            "unconditional_response_fault_detection",
            len(killed_faults),
            len(fault_ids),
        ),
    ]

    unavailable = dict(evidence.get("unavailable_endpoints") or {})
    if evidence.get("collection_status") == "EVALUATOR_UNAVAILABLE":
        unavailable.update({name: "COLLECTION_EVALUATOR_UNAVAILABLE"
                            for name in PRIMARY_ENDPOINTS if name not in unavailable})
    elif evidence.get("baseline_status") == "EVALUATOR_UNAVAILABLE":
        for name in PRIMARY_ENDPOINTS:
            if name != "collection_success":
                unavailable.setdefault(name, "BASELINE_EVALUATOR_UNAVAILABLE")
    if any(not reason or name not in PRIMARY_ENDPOINTS
           for name, reason in unavailable.items()):
        raise ValueError("Invalid endpoint unavailability evidence")
    for row in results:
        if row["endpoint"] in unavailable:
            row.update(numerator=None, score=None, observed=False,
                       literal_denominator_observed=False,
                       failure_reason=unavailable[row["endpoint"]])
    return results


def paired_metric_differences(
    l0_scores,
    l1_scores,
):
    l0 = {
        row["endpoint"]: row
        for row in l0_scores
    }

    l1 = {
        row["endpoint"]: row
        for row in l1_scores
    }

    if (
        set(l0) != set(PRIMARY_ENDPOINTS)
        or set(l1) != set(PRIMARY_ENDPOINTS)
    ):
        raise ValueError(
            "Incomplete seven-endpoint score vectors."
        )

    differences = {}

    for endpoint in PRIMARY_ENDPOINTS:
        left = l0[endpoint]["score"]
        right = l1[endpoint]["score"]

        differences[endpoint] = (
            None
            if left is None or right is None
            else right - left
        )

    return differences


# END_STAGE06_METRICS_MODULE

from pathlib import Path
import importlib
import json
import sys
import time


CELL7_STARTED = time.time()

require_cell_pass(6)


def freeze_visible_module(
    module_name: str,
    destination: Path,
) -> str:
    """
    Persist the explicitly marked source region of this cell.

    The saved module contains the actual visible function
    definitions, never a quoted/escaped copy of the module.
    """
    shell = get_ipython()

    if shell is None:
        raise RuntimeError(
            "A Jupyter/IPython kernel is required."
        )

    history = shell.history_manager.input_hist_raw

    start_marker = (
        f"# BEGIN_STAGE06_{module_name}_MODULE"
    )

    end_marker = (
        f"# END_STAGE06_{module_name}_MODULE"
    )

    candidates = [
        source for source in reversed(history[-12:])
        if start_marker in source and end_marker in source
    ]
    if not candidates:
        raise RuntimeError(
            "Cannot recover this cell source from IPython history; "
            "run the complete cell in Kaggle."
        )
    cell_source = candidates[0]
    if cell_source.count(start_marker) != 1 or cell_source.count(end_marker) != 1:
        raise RuntimeError("Ambiguous source-region markers.")

    source = (
        cell_source.split(
            start_marker,
            1,
        )[1].split(
            end_marker,
            1,
        )[0].strip()
        + "\n"
    )

    return write_immutable(
        destination,
        source,
    )


metrics_module_path = (
    PROJECT_ROOT
    / "src/metrics.py"
)

metrics_write_status = freeze_visible_module(
    "METRICS",
    metrics_module_path,
)

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT / "src"),
    )

importlib.invalidate_caches()

import metrics as frozen_metrics

if len(frozen_metrics.PRIMARY_ENDPOINTS) != 7:
    raise RuntimeError(
        "Metrics module does not define seven endpoints."
    )

cell7_report = save_cell_report(
    7,
    metrics_module_sha256=sha256_file(
        metrics_module_path
    ),
    module_integrity="HASH_VERIFIED",
    public_functions=[
        "score_operation_condition",
        "paired_metric_differences",
    ],
)

print(
    json.dumps(
        cell7_report,
        indent=2,
    )
)

print("\nNOTEBOOK 06 — CELL 7: PASS")

{
  "metrics_module_sha256": "e4b8c94782c25d8e6a0b4baa2caeccc38c69bb0fb595c69b50af41655245fc25",
  "module_integrity": "HASH_VERIFIED",
  "public_functions": [
    "score_operation_condition",
    "paired_metric_differences"
  ],
  "status": "PASS",
  "notebook": "06",
  "cell": 7
}

NOTEBOOK 06 — CELL 7: PASS


In [8]:
# NOTEBOOK 06 — CELL 8
# Qualify all seven scoring rules against development fixtures.

from __future__ import annotations

from copy import deepcopy
import json
import time

import metrics


CELL8_STARTED = time.time()

require_cell_pass(7)

fixture_oracle = {
    "scenarios": [
        {
            "scenario_id": "s1",
            "applicability": "APPLICABLE",
        }
    ],
    "requirements": [
        {
            "requirement_id": "r1",
            "applicability": "APPLICABLE",
        }
    ],
    "baselines": [
        {
            "baseline_id": "b1",
        }
    ],
    "faults": [
        {
            "fault_id": "f1",
            "applicable": True,
        }
    ],
}

qualified_fixture_results = []


def record_fixture(
    name,
    passed,
):
    qualified_fixture_results.append({
        "fixture": name,
        "status": (
            "PASS"
            if passed
            else "FAIL"
        ),
    })

    if not passed:
        raise RuntimeError(
            "Metric qualification failed: "
            + name
        )


empty_evidence = {
    "collection_status": "MISSING_OUTPUT",
}

empty_scores = metrics.score_operation_condition(
    empty_evidence,
    fixture_oracle,
)

record_fixture(
    "missing_output",
    [
        row["score"]
        for row in empty_scores
    ] == [
        0,
        0,
        None,
        0,
        0,
        0,
        0,
    ],
)

record_fixture(
    "no_claims_literal_denominator",
    empty_scores[4]["denominator"] == 0
    and empty_scores[4]["score"] == 0,
)

for failure in (
    "COMPILE_ERROR",
    "COLLECTION_ERROR",
    "NO_TESTS",
    "SAFETY_REJECTED",
    "TIMEOUT",
):
    scores = metrics.score_operation_condition(
        {
            "collection_status": failure,
        },
        fixture_oracle,
    )

    record_fixture(
        failure.lower(),
        scores[0]["score"] == 0
        and scores[6]["denominator"] == 1,
    )


for outcomes in (
    ["skipped"],
    ["xfailed"],
    ["failed"],
    [],
):
    scores = metrics.score_operation_condition(
        {
            "collection_status": "PASS",
            "collected_items": 1,
            "baseline_status": "COMPLETE",
            "item_outcomes": outcomes,
        },
        fixture_oracle,
    )

    record_fixture(
        "baseline_" + (
            outcomes[0]
            if outcomes
            else "no_normal_pass"
        ),
        scores[1]["score"] == 0,
    )


qualified_evidence = {
    "collection_status": "PASS",
    "collected_items": 1,
    "baseline_status": "COMPLETE",
    "item_outcomes": ["passed"],
    "request_attempts": [
        {
            "attempt_id": "attempt1",
            "classifiable": True,
            "matched_intent": True,
            "scenario_id": "s1",
        }
    ],
    "reviewed_claims": [
        {
            "reviewed": True,
            "supported": True,
        }
    ],
    "effective_checks": [
        {
            "requirement_id": "r1",
            "reached": True,
            "grounded": True,
            "linkage_qualified": True,
        }
    ],
    "mutation_results": [
        {
            "fault_id": "f1",
            "result": "KILLED",
            "consumed": True,
        }
    ],
}

qualified_scores = (
    metrics.score_operation_condition(
        qualified_evidence,
        fixture_oracle,
    )
)

record_fixture(
    "all_seven_success",
    all(
        row["score"] == 1
        for row in qualified_scores
    ),
)

not_consumed = deepcopy(
    qualified_evidence
)

not_consumed["mutation_results"] = [
    {
        "fault_id": "f1",
        "result": "NOT_CONSUMED",
        "consumed": False,
    }
]

record_fixture(
    "fault_not_consumed",
    metrics.score_operation_condition(
        not_consumed,
        fixture_oracle,
    )[6]["score"] == 0,
)

survived = deepcopy(
    qualified_evidence
)

survived["mutation_results"] = [
    {
        "fault_id": "f1",
        "result": "SURVIVED",
        "consumed": True,
    }
]

record_fixture(
    "fault_survival",
    metrics.score_operation_condition(
        survived,
        fixture_oracle,
    )[6]["score"] == 0,
)

baseline_failure = deepcopy(
    qualified_evidence
)

baseline_failure["item_outcomes"] = [
    "failed"
]

record_fixture(
    "baseline_failure_cancels_kill",
    metrics.score_operation_condition(
        baseline_failure,
        fixture_oracle,
    )[6]["score"] == 0,
)

for denominator_name in (
    "requirements",
    "scenarios",
    "faults",
    "baselines",
):
    bad_oracle = deepcopy(
        fixture_oracle
    )

    bad_oracle[denominator_name] = []

    rejected = False

    try:
        metrics.score_operation_condition(
            {},
            bad_oracle,
        )

    except ValueError:
        rejected = True

    record_fixture(
        "zero_" + denominator_name,
        rejected,
    )


evaluator_error_rejected = False

try:
    metrics.score_operation_condition(
        {
            "evaluator_error": "BROKEN_ROUTER",
        },
        fixture_oracle,
    )

except RuntimeError:
    evaluator_error_rejected = True

record_fixture(
    "evaluator_error_not_model_failure",
    evaluator_error_rejected,
)

condition_independence = deepcopy(
    qualified_evidence
)

condition_independence["condition"] = "L0"

l0 = metrics.score_operation_condition(
    condition_independence,
    fixture_oracle,
)

condition_independence["condition"] = "L1"

l1 = metrics.score_operation_condition(
    condition_independence,
    fixture_oracle,
)

record_fixture(
    "condition_blind_correctness",
    l0 == l1,
)

differences = metrics.paired_metric_differences(
    l0,
    l1,
)

record_fixture(
    "paired_difference_direction",
    all(
        value == 0
        for value in differences.values()
        if value is not None
    ),
)

unavailable_scores = metrics.score_operation_condition(
    {**qualified_evidence, "baseline_status": "EVALUATOR_UNAVAILABLE"},
    fixture_oracle,
)
record_fixture(
    "baseline_evaluator_limitation_preserves_collection_and_withholds_affected_scores",
    unavailable_scores[0]["score"] == 1
    and all(row["score"] is None and not row["observed"]
            for row in unavailable_scores[1:]),
)
fixture_report = {
    "status": "PASS",
    "fixture_count": len(
        qualified_fixture_results
    ),
    "fixtures": qualified_fixture_results,
    "module_sha256": sha256_file(
        PROJECT_ROOT
        / "src/metrics.py"
    ),
}

write_json_immutable(
    PROJECT_ROOT
    / "reports/qualification/metrics_fixtures.json",
    fixture_report,
)

cell8_report = save_cell_report(
    8,
    metric_fixture_count=len(
        qualified_fixture_results
    ),
    qualification_status="PASS",
)

print(
    json.dumps(
        cell8_report,
        indent=2,
    )
)

print("\nNOTEBOOK 06 — CELL 8: PASS")

{
  "metric_fixture_count": 23,
  "qualification_status": "PASS",
  "status": "PASS",
  "notebook": "06",
  "cell": 8
}

NOTEBOOK 06 — CELL 8: PASS


In [9]:
# NOTEBOOK 06 — CELL 9
# Implement the frozen offline request classifier,
# normalization policy, and replay response selector.

# BEGIN_STAGE06_ROUTING_MODULE

"""Deterministic OpenAPI request evaluation and frozen response delivery."""
from __future__ import annotations

import base64
import hashlib
import json
import re
from urllib.parse import parse_qs, unquote, urlsplit

from jsonschema import Draft4Validator, Draft202012Validator
from jsonschema.exceptions import SchemaError
from referencing import Registry, Resource
from referencing.jsonschema import DRAFT4, DRAFT202012


ROUTING_VERSION = "stage06-openapi-contract-v6"
DOCUMENT_URI = "urn:stage06:canonical-operation"
FORM_MEDIA_TYPE = "application/x-www-form-urlencoded"
LEGACY_FORM_RULE = {
    "op_id": "op_42818f46abfce2a2ec0520ea",
    "method": "PUT",
    "path_template": "/api/v1/filters/{id}",
    "document_sha256": "0f23e5b6c3e149d86ba4e3920ad0cb9f3a5feee81ce14b32f839852cb8b56cff",
    "declared_media_type": "application/form-data",
    "wire_media_type": FORM_MEDIA_TYPE,
    "array_fields": {"context": "context[]"},
    "sources": [
        "https://docs.joinmastodon.org/methods/filters/#update-legacy",
        "https://docs.joinmastodon.org/client/intro/#form-data",
        "https://docs.joinmastodon.org/client/intro/#array",
    ],
}
NEGATIVE_INTENTS = {
    "invalid_missing_required_parameter": ("parameter", "missing"),
    "invalid_missing_required_request_body": ("body", "missing"),
    "parameter_invalid_wrong_type": ("parameter", "type"),
    "parameter_invalid_enum_value": ("parameter", "enum"),
    "parameter_invalid_below_minimum": ("parameter", "minimum"),
    "parameter_invalid_below_minimum_items": ("parameter", "minItems"),
    "body_invalid_wrong_type": ("body", "type"),
    "body_invalid_missing_required_property": ("body", "required"),
    "body_invalid_below_minimum": ("body", "minimum"),
    "body_invalid_below_minimum_items": ("body", "minItems"),
    "body_invalid_below_minimum_length": ("body", "minLength"),
    "body_invalid_above_maximum_length": ("body", "maxLength"),
}


class UnsupportedRoute(RuntimeError):
    failure_owner = "GENERATED_OUTPUT"

    def __init__(self, code, detail=None):
        self.code = code
        self.detail = detail
        super().__init__(code)


class EvaluatorUnavailable(UnsupportedRoute):
    failure_owner = "EVALUATOR_LIMITATION"


class EvaluatorDefect(UnsupportedRoute):
    failure_owner = "EVALUATOR_FAILURE"


def normalize_prepared_request(method, url, headers=None, body=None):
    """Keep transport details even for requests the router will reject."""
    try:
        parts = urlsplit(url)
        origin_allowed = (
            parts.scheme == "http" and parts.netloc.lower() == "openapi.test"
        )
    except ValueError as error:
        raise UnsupportedRoute("MALFORMED_REQUEST_URL") from error
    headers = {str(k).lower(): str(v) for k, v in (headers or {}).items()}
    cookies = dict(
        tuple(part.strip().split("=", 1))
        for part in headers.get("cookie", "").split(";") if "=" in part
    )
    content_type = headers.get("content-type", "").lower().strip()
    media_type = content_type.split(";", 1)[0].strip()
    body_present = body is not None and body != b"" and body != ""
    body_encoding = "none"
    if body_present:
        if isinstance(body, bytes):
            try:
                body = body.decode("utf-8")
            except UnicodeDecodeError:
                body = {"base64": base64.b64encode(body).decode("ascii")}
                body_encoding = "binary"
        if isinstance(body, str):
            body_encoding = "text"
            if _is_json(media_type):
                try:
                    body = json.loads(body, parse_constant=_reject_json_constant)
                    body_encoding = "json"
                except ValueError:
                    body_encoding = "malformed_json"
            elif media_type == "application/x-www-form-urlencoded":
                body = _query(body)
                body_encoding = "form"
        elif body_encoding != "binary":
            body_encoding = "json"
    else:
        body = None
    return {
        "method": method.upper(), "raw_url": url, "origin_allowed": origin_allowed,
        "path": parts.path or "/", "query": _query(parts.query),
        "headers": headers, "cookies": cookies, "body": body,
        "body_present": body_present, "body_encoding": body_encoding,
        "media_type": content_type,
    }


def _is_json(media_type):
    return media_type in {"application/json", "text/json"} or media_type.endswith("+json")


def _reject_json_constant(value):
    raise ValueError(f"Non-JSON constant: {value}")


def _query(text):
    return {key: values[0] if len(values) == 1 else values
            for key, values in parse_qs(text, keep_blank_values=True).items()}


def match_path_template(template, actual_path):
    """Match path segments before unquoting parameter values."""
    names = re.findall(r"\{([^}]+)\}", template)
    fragments = re.split(r"\{[^}]+\}", template)
    pattern = "([^/]+)".join(re.escape(fragment) for fragment in fragments)
    match = re.fullmatch(pattern, actual_path)
    return None if match is None else dict(zip(names, map(unquote, match.groups())))


def _local_reference(document, reference):
    if not isinstance(reference, str) or not reference.startswith("#/"):
        raise EvaluatorUnavailable("UNSUPPORTED_REFERENCE", reference)
    node = document
    try:
        for part in unquote(reference[2:]).split("/"):
            node = node[part.replace("~1", "/").replace("~0", "~")]
    except (KeyError, TypeError) as error:
        raise EvaluatorUnavailable("UNRESOLVED_LOCAL_REFERENCE", reference) from error
    return node


def _object(document, value):
    seen = set()
    while isinstance(value, dict) and "$ref" in value:
        reference = value["$ref"]
        if reference in seen:
            raise EvaluatorUnavailable("CYCLIC_OBJECT_REFERENCE")
        seen.add(reference)
        value = _local_reference(document, reference)
    return value


def _schema_tree(value, openapi30):
    if isinstance(value, list):
        return [_schema_tree(item, openapi30) for item in value]
    if not isinstance(value, dict):
        return value
    result = {
        key: (item if key in {"enum", "const", "default", "example", "examples"}
              else _schema_tree(item, openapi30))
        for key, item in value.items()
    }
    if openapi30 and result.get("nullable") is True and isinstance(result.get("type"), str):
        result["type"] = [result["type"], "null"]
    return result


def _schema_errors(schema, instance, document):
    if not isinstance(schema, (dict, bool)):
        raise EvaluatorDefect("MALFORMED_FROZEN_SCHEMA")
    openapi30 = document.get("openapi", "").startswith("3.0.")
    spec = DRAFT4 if openapi30 else DRAFT202012
    validator = Draft4Validator if openapi30 else Draft202012Validator
    if openapi30 and isinstance(schema, bool):
        raise EvaluatorUnavailable("UNSUPPORTED_OPENAPI30_BOOLEAN_SCHEMA")
    # A registry without a retrieval callback only resolves the supplied
    # canonical document and built-in meta-schemas. It cannot fetch URLs.
    local_root = _schema_tree(document, openapi30)
    registry = Registry().with_resource(DOCUMENT_URI, Resource(local_root, spec))
    schema = _schema_tree(schema, openapi30)
    try:
        validator.check_schema(schema)
        return list(validator(schema, registry=registry,
                              _resolver=registry.resolver(DOCUMENT_URI),
                              format_checker=validator.FORMAT_CHECKER).iter_errors(instance))
    except SchemaError as error:
        raise EvaluatorUnavailable("UNSUPPORTED_FROZEN_SCHEMA", str(error)) from error
    except Exception as error:
        # Referencing exceptions are wrapped by jsonschema.
        if type(error).__module__.split(".")[0] in {"jsonschema", "referencing"}:
            raise EvaluatorUnavailable("UNRESOLVED_SCHEMA_REFERENCE", str(error)) from error
        raise


def _scalar(value, schema):
    kind = schema.get("type")
    if isinstance(kind, list):
        choices = [item for item in kind if item != "null"]
        if len(choices) != 1:
            raise EvaluatorUnavailable("AMBIGUOUS_PARAMETER_SERIALIZATION")
        kind = choices[0]
    if kind == "integer" and re.fullmatch(r"[+-]?[0-9]+", value):
        try:
            return int(value)
        except ValueError:
            return value
    if kind == "number" and re.fullmatch(r"[+-]?(?:[0-9]+(?:\.[0-9]*)?|\.[0-9]+)(?:[eE][+-]?[0-9]+)?", value):
        number = float(value)
        if number not in (float("inf"), float("-inf")):
            return number
    if kind == "boolean" and value in {"true", "false"}:
        return value == "true"
    return value


def _parameter_value(raw, parameter, schema, document):
    location = parameter["in"]
    style = parameter.get("style", "form" if location in {"query", "cookie"} else "simple")
    explode = parameter.get("explode", style == "form")
    if style not in ({"form", "spaceDelimited", "pipeDelimited"} if location == "query"
                     else {"form"} if location == "cookie" else {"simple"}):
        raise EvaluatorUnavailable("UNSUPPORTED_PARAMETER_STYLE", (location, style))
    if schema.get("type") == "object":
        if explode:
            raise EvaluatorUnavailable("UNSUPPORTED_EXPLODED_OBJECT_PARAMETER", parameter["name"])
        if isinstance(raw, list):
            raise EvaluatorUnavailable("AMBIGUOUS_OBJECT_PARAMETER", parameter["name"])
        parts = raw.split(",")
        if len(parts) % 2:
            raise EvaluatorUnavailable("UNSUPPORTED_OBJECT_PARAMETER", parameter["name"])
        return {key: _scalar(value, schema.get("properties", {}).get(key, {}))
                for key, value in zip(parts[::2], parts[1::2])}
    if schema.get("type") != "array":
        if isinstance(raw, list):
            return raw
        return _scalar(raw, schema)
    items = _object(document, schema.get("items") or {})
    if style == "form" and explode:
        values = raw if isinstance(raw, list) else [raw]
    else:
        if isinstance(raw, list):
            raise EvaluatorUnavailable("AMBIGUOUS_ARRAY_SERIALIZATION", parameter["name"])
        values = raw.split({"spaceDelimited": " ", "pipeDelimited": "|"}.get(style, ","))
    return [_scalar(value, items) for value in values]


def _select_media(content, received):
    if not received:
        raise EvaluatorUnavailable("REQUEST_MEDIA_TYPE_UNSPECIFIED")
    received, *parameters = [part.strip() for part in received.split(";")]
    received_params = set(parameters)
    matches = []
    for declared, media in content.items():
        name, *params = [part.strip() for part in declared.lower().split(";")]
        if not set(params).issubset(received_params):
            continue
        if name == received:
            matches.append(((2, len(params)), media))
        elif name == received.split("/", 1)[0] + "/*":
            matches.append(((1, len(params)), media))
        elif name == "*/*":
            matches.append(((0, len(params)), media))
    if not matches:
        return None
    best = max(rank for rank, _ in matches)
    winners = [media for rank, media in matches if rank == best]
    if len(winners) != 1:
        raise EvaluatorUnavailable("AMBIGUOUS_REQUEST_MEDIA_TYPE")
    return winners[0]


def _operation(document, template, method):
    try:
        path = _object(document, document["paths"][template])
        return path, _object(document, path[method.lower()])
    except (KeyError, TypeError) as error:
        raise EvaluatorDefect("ORACLE_OPERATION_NOT_IN_CANONICAL_DOCUMENT") from error


SUCCESS_STATUS_POLICY = "single-explicit-success-v1"


def success_status_mapping(document, path_template, method):
    """Return a status only when the operation has one explicit success candidate."""
    _, operation = _operation(document, path_template, method)
    responses = operation.get("responses", {})
    codes = sorted(str(code) for code in responses if re.fullmatch(r"2[0-9]{2}", str(code)))
    has_range = any(str(code).upper() == "2XX" for code in responses)
    if len(codes) > 1 or has_range:
        raise EvaluatorUnavailable("AMBIGUOUS_SUCCESS_STATUS_MAPPING",
                                   {"statuses": codes, "has_2xx_range": has_range})
    return {"status": int(codes[0])} if codes else None


def verify_success_status_mapping(document, scenario):
    if scenario["expected_validity"] != "VALID":
        return
    expected = success_status_mapping(document, scenario["path_template"], scenario["method"])
    if expected is None:
        raise EvaluatorUnavailable("NO_DOCUMENTED_RESPONSE_MAPPING")
    if scenario.get("documented_response_mapping") != expected:
        raise EvaluatorDefect("FROZEN_SUCCESS_STATUS_MAPPING_MISMATCH")


def legacy_form_rule(document, template, method):
    rule = LEGACY_FORM_RULE
    if (template, method.upper()) != (rule["path_template"], rule["method"]):
        return None
    encoded = json.dumps(document, sort_keys=True, separators=(",", ":"),
                         ensure_ascii=False).encode()
    return rule if hashlib.sha256(encoded).hexdigest() == rule["document_sha256"] else None


def _decode_form(body, media, document, rule):
    schema = _object(document, media.get("schema", {}))
    if media.get("encoding") or not isinstance(body, dict):
        raise EvaluatorUnavailable("UNSUPPORTED_FORM_ENCODING")
    properties = schema.get("properties", {})
    aliases = {wire: logical for logical, wire in (rule or {}).get("array_fields", {}).items()}
    decoded = {}
    for wire, raw in body.items():
        if wire in (rule or {}).get("array_fields", {}):
            raise EvaluatorUnavailable("UNSUPPORTED_UNBRACKETED_FORM_ARRAY", wire)
        logical = aliases.get(wire, wire)
        property_schema = _object(document, properties.get(logical, {}))
        decoded[logical] = _parameter_value(
            raw, {"in": "query", "name": logical}, property_schema, document)
    return decoded


def _security_observation(document, operation, request):
    """Record credential presence; never infer authentication success offline."""
    alternatives = operation.get("security", document.get("security", []))
    if not alternatives or {} in alternatives:
        return "NOT_REQUIRED"
    schemes = (document.get("components") or {}).get("securitySchemes", {})
    for option in alternatives:
        supplied = []
        for name in option:
            scheme = _object(document, schemes.get(name, {}))
            if scheme.get("type") == "apiKey":
                location, key = scheme.get("in"), scheme.get("name", "")
                source = {"header": request["headers"], "query": request["query"],
                          "cookie": request["cookies"]}.get(location, {})
                supplied.append(bool(source.get(key.lower() if location == "header" else key)))
            else:
                supplied.append(bool(request["headers"].get("authorization")))
        if supplied and all(supplied):
            return "PRESENT_NOT_VERIFIED"
    return "MISSING_OR_UNVERIFIABLE"


def _assess(request, document, template, path_values):
    path, operation = _operation(document, template, request["method"])
    parameters = {}
    for raw in [*path.get("parameters", []), *operation.get("parameters", [])]:
        parameter = _object(document, raw)
        location, name = parameter.get("in"), parameter.get("name")
        if location not in {"path", "query", "header", "cookie"} or not name:
            raise EvaluatorUnavailable("UNSUPPORTED_PARAMETER_DEFINITION")
        key = name.lower() if location == "header" else name
        parameters[location, key] = parameter
    values = {"path": path_values, "query": request["query"],
              "header": request["headers"], "cookie": request["cookies"]}
    violations = set()
    typed = {}
    for (location, key), parameter in parameters.items():
        if key not in values[location]:
            if parameter.get("required") is True or location == "path":
                violations.add(("parameter", location, key, "missing"))
            continue
        if "content" in parameter:
            raise EvaluatorUnavailable("UNSUPPORTED_PARAMETER_CONTENT", (location, key))
        schema = parameter.get("schema") or {}
        effective_schema = _object(document, schema)
        value = _parameter_value(values[location][key], parameter, effective_schema, document)
        typed.setdefault(location, {})[key] = value
        for error in _schema_errors(schema, value, document):
            violations.add(("parameter", location, key, str(error.validator)))
        if (isinstance(value, int) and not isinstance(value, bool)
                and effective_schema.get("format") in {"int32", "int64"}):
            bits = 32 if effective_schema["format"] == "int32" else 64
            if not -(2 ** (bits - 1)) <= value < 2 ** (bits - 1):
                violations.add(("parameter", location, key, "format"))
    request_body = _object(document, operation.get("requestBody", {}))
    if not request["body_present"]:
        if request_body.get("required") is True:
            violations.add(("body", "", "", "missing"))
    elif request_body:
        content = request_body.get("content", {})
        rule = legacy_form_rule(document, template, request["method"])
        wire_type = request["media_type"].split(";", 1)[0].strip()
        media = (_select_media(content, rule["declared_media_type"])
                 if rule and wire_type == rule["wire_media_type"]
                 else _select_media(content, request["media_type"]))
        if rule and wire_type == rule["declared_media_type"]:
            raise EvaluatorUnavailable("UNSUPPORTED_LEGACY_FORM_WIRE_TYPE")
        if media is None:
            violations.add(("body", "", "", "media_type"))
        elif request["body_encoding"] == "malformed_json":
            violations.add(("body", "", "", "malformed_json"))
        else:
            if request["body_encoding"] not in {"json", "form"}:
                raise EvaluatorUnavailable("UNSUPPORTED_REQUEST_BODY_SERIALIZATION")
            value = request["body"]
            schema = media.get("schema", {})
            if request["body_encoding"] == "form":
                value = _decode_form(value, media, document, rule)
            for error in _schema_errors(schema, value, document):
                pointer = "/".join(map(str, error.absolute_path))
                if error.validator == "required" and isinstance(error.instance, dict):
                    for missing in set(error.validator_value) - error.instance.keys():
                        violations.add(("body", "", f"{pointer}/{missing}", "required"))
                else:
                    violations.add(("body", "", pointer, str(error.validator)))
    return typed, violations, _security_observation(document, operation, request)


def _literal(value):
    if isinstance(value, bool):
        return str(value).lower()
    return str(value)


def _fixture_values(scenario):
    values = scenario.get("parameter_values") or {}
    return {
        location: {
            key.lower() if location == "header" else key:
            ([_literal(x) for x in value] if isinstance(value, list) and len(value) != 1
             else _literal(value[0]) if isinstance(value, list) and len(value) == 1
             else _literal(value))
            for key, value in (values.get(location) or {}).items()
        }
        for location in ("path", "query", "header", "cookie")
    }


def _exact_fixture(request, path, scenario):
    values = _fixture_values(scenario)
    return (
        path == values["path"] and request["query"] == values["query"]
        and all(request["headers"].get(k) == v for k, v in values["header"].items())
        and all(request["cookies"].get(k) == v for k, v in values["cookie"].items())
        and (scenario.get("body") is not None) == request["body_present"]
        and json.dumps(request["body"], sort_keys=True)
        == json.dumps(scenario.get("body"), sort_keys=True)
    )


def _negative_signature(scenario, document, template, request):
    values = _fixture_values(scenario)
    fixture_request = {
        **request, "query": values["query"], "headers": values["header"],
        "cookies": values["cookie"], "body": scenario.get("body"),
        "body_present": scenario.get("body") is not None, "body_encoding": "json",
    }
    try:
        _, violations, _ = _assess(fixture_request, document, template, values["path"])
    except EvaluatorUnavailable:
        return set()
    return violations


def prepare_frozen_fixture(document, scenario):
    """Serialize a frozen positive fixture according to its operation."""
    from urllib.parse import quote, urlencode

    template = scenario["path_template"]
    path, operation = _operation(document, template, scenario["method"])
    parameters = {}
    for raw in [*path.get("parameters", []), *operation.get("parameters", [])]:
        parameter = _object(document, raw)
        parameters[parameter["in"], parameter["name"].lower()
                   if parameter["in"] == "header" else parameter["name"]] = parameter
    values = _fixture_values(scenario)
    for location, entries in (scenario.get("parameter_values") or {}).items():
        for name, value in entries.items():
            key = name.lower() if location == "header" else name
            parameter = parameters.get((location, key), {})
            style = parameter.get("style", "form" if location in {"query", "cookie"} else "simple")
            explode = parameter.get("explode", style == "form")
            if isinstance(value, dict) and not value and style == "form" and explode:
                # An empty exploded object produces no wire parameters.
                values[location].pop(key, None)
            elif isinstance(value, list) and (style != "form" or not explode):
                separator = {"spaceDelimited": " ", "pipeDelimited": "|"}.get(style, ",")
                values[location][key] = separator.join(map(_literal, value))
    url_path = template
    for key, value in values["path"].items():
        url_path = url_path.replace("{" + key + "}", quote(value, safe=""))
    if "{" in url_path:
        raise EvaluatorUnavailable("FROZEN_FIXTURE_PATH_UNSERIALIZABLE")
    headers = values["header"]
    if values["cookie"]:
        headers["cookie"] = "; ".join(f"{k}={v}" for k, v in values["cookie"].items())
    body = scenario.get("body")
    if body is not None:
        content = _object(document, operation.get("requestBody", {})).get("content", {})
        json_media = [m for m in content if _is_json(m.split(";", 1)[0].strip().lower())]
        rule = legacy_form_rule(document, template, scenario["method"])
        if rule and rule["declared_media_type"] in content and isinstance(body, dict):
            headers["content-type"] = rule["wire_media_type"]
            form = {}
            for name, value in body.items():
                wire = rule["array_fields"].get(name, name)
                form[wire] = ([_literal(item) for item in value]
                              if isinstance(value, list) else _literal(value))
            encoded = urlencode(form, doseq=True)
            decoded = _decode_form(_query(encoded), content[rule["declared_media_type"]],
                                   document, rule)
            if decoded != body:
                raise EvaluatorDefect("FROZEN_FORM_ROUNDTRIP_MISMATCH")
            body = encoded.encode()
        elif json_media:
            declared = headers.get("content-type", "")
            if declared not in json_media:
                headers["content-type"] = min(
                    json_media, key=lambda media: (media != "application/json", media)
                )
            body = json.dumps(body).encode()
        elif "application/x-www-form-urlencoded" in content and isinstance(body, dict):
            headers["content-type"] = "application/x-www-form-urlencoded"
            body = urlencode(body, doseq=True).encode()
        else:
            raise EvaluatorUnavailable("FROZEN_FIXTURE_BODY_UNSERIALIZABLE", sorted(content))
    query = urlencode(values["query"], doseq=True)
    return normalize_prepared_request(
        scenario["method"], "http://openapi.test" + url_path + ("?" + query if query else ""),
        headers, body,
    )


def prepare_frozen_response(record, baseline):
    """Serialize JSON strings as JSON and preserve registered response headers."""
    evidence = record.get("source_evidence") or ()
    if isinstance(evidence, dict):
        evidence = [evidence]
    baseline_evidence = baseline.get("source_evidence") or ()
    if isinstance(baseline_evidence, dict):
        baseline_evidence = [baseline_evidence]
    media_type = next((row["media_type"] for row in [*evidence, *baseline_evidence]
                       if row.get("media_type")), "")
    headers = dict(record.get("headers") or {})
    content_type = next((v for k, v in headers.items() if k.lower() == "content-type"), "")
    body = record.get("body")
    if body is None:
        return headers, b""
    if not content_type and media_type:
        headers["Content-Type"] = media_type
        content_type = media_type
    if _is_json(content_type.split(";", 1)[0].strip().lower()) or not isinstance(body, str):
        return headers, json.dumps(body, allow_nan=False).encode()
    return headers, body.encode()


def audit_frozen_response(document, scenario, record, baseline):
    """Independently check that a baseline or variation obeys its response contract."""
    verify_success_status_mapping(document, scenario)
    _, operation = _operation(document, scenario["path_template"], scenario["method"])
    code = str(record["status"])
    responses = operation.get("responses", {})
    response = responses.get(code, responses.get(code[:1] + "XX", responses.get("default")))
    if response is None:
        raise EvaluatorDefect("UNDOCUMENTED_FROZEN_RESPONSE_STATUS", code)
    response = _object(document, response)
    headers, _ = prepare_frozen_response(record, baseline)
    headers = {key.lower(): value for key, value in headers.items()}
    for name, raw in (response.get("headers") or {}).items():
        declaration = _object(document, raw)
        if declaration.get("required") and name.lower() not in headers:
            raise EvaluatorDefect("MISSING_REQUIRED_FROZEN_RESPONSE_HEADER", name)
    content = response.get("content") or {}
    if not content:
        if record.get("body") is not None:
            raise EvaluatorDefect("UNDOCUMENTED_FROZEN_RESPONSE_BODY")
        return
    media = _select_media(content, headers.get("content-type", ""))
    if media is None:
        raise EvaluatorDefect("FROZEN_RESPONSE_MEDIA_TYPE_MISMATCH")
    errors = _schema_errors(media.get("schema", {}), record.get("body"), document)
    if errors:
        raise EvaluatorDefect("FROZEN_RESPONSE_SCHEMA_MISMATCH",
                              sorted({str(error.validator) for error in errors}))


def classify_request(request, frozen_scenarios, operation_document=None):
    if not request["origin_allowed"]:
        raise UnsupportedRoute("UNAUTHORIZED_HOST")
    matches = []
    for scenario in frozen_scenarios:
        if (scenario.get("applicability") == "APPLICABLE"
                and request["method"] == scenario["method"].upper()):
            template = scenario["path_template"]
            path = match_path_template(template, request["path"])
            if path is not None:
                matches.append((scenario, template, path))
    if not matches:
        raise UnsupportedRoute("REQUEST_OUTSIDE_TARGET_OPERATION")
    if len({scenario["scenario_id"] for scenario, _, _ in matches}) != len(matches):
        raise EvaluatorDefect("DUPLICATE_REGISTERED_SCENARIO")
    if len({template for _, template, _ in matches}) != 1:
        raise EvaluatorUnavailable("AMBIGUOUS_OPERATION_TEMPLATE")
    if operation_document is None:
        raise EvaluatorDefect("MISSING_CANONICAL_OPERATION_DOCUMENT")
    _, template, path = matches[0]
    typed, violations, security = _assess(request, operation_document, template, path)
    candidates = []
    if violations:
        for scenario, _, _ in matches:
            if scenario["expected_validity"] != "INVALID":
                continue
            signature = _negative_signature(scenario, operation_document, template, request)
            # An invalid fixture with multiple discrepancies does not define
            # a single-violation scenario that the evaluator can credit.
            if len(signature) == 1 and signature == violations:
                scope, _, _, validator = next(iter(signature))
                if NEGATIVE_INTENTS.get(scenario["intent"]) == (scope, validator):
                    candidates.append(scenario)
        exact = [s for s in candidates if _exact_fixture(request, path, s)]
        candidates = exact or candidates
        status = "SEMANTIC_NEGATIVE_MATCH"
    else:
        valid = [s for s, _, _ in matches if s["expected_validity"] == "VALID"]
        exact = [s for s in valid if _exact_fixture(request, path, s)]
        candidates = exact or [s for s in valid if s["intent"] == "valid_baseline_request"]
        status = "EXACT_FROZEN_SCENARIO_REPLAY" if exact else "SEMANTIC_VALID_BASELINE"

    if len(candidates) != 1:
        raise EvaluatorUnavailable("AMBIGUOUS_OR_UNMAPPED_REQUEST_SEMANTICS", {
            "candidate_ids": [s["scenario_id"] for s in candidates],
            "observed_violations": sorted(violations),
        })
    matched = candidates[0]
    verify_success_status_mapping(operation_document, matched)
    return {
        "classification_status": status, "classifiable": True,
        "scenario_id": matched["scenario_id"], "intent": matched["intent"],
        "expected_validity": matched["expected_validity"], "positive": not violations,
        "documented_response_mapping": matched.get("documented_response_mapping"),
        "typed_parameters": typed, "observed_violations": sorted(violations),
        "security_observation": security,
    }


def select_frozen_response(classification, baselines, mode="baseline", mutation=None):
    scenario_id = classification["scenario_id"]
    mapping = classification.get("documented_response_mapping")
    if mapping is None:
        raise EvaluatorUnavailable("NO_DOCUMENTED_RESPONSE_MAPPING")
    linked = [row for row in baselines if row["scenario_id"] == scenario_id]
    if len(linked) != 1:
        raise EvaluatorDefect("FROZEN_BASELINE_LINK_COUNT", len(linked))
    baseline = linked[0]
    if int(baseline["status"]) != int(mapping["status"]):
        raise EvaluatorDefect("FROZEN_BASELINE_STATUS_MISMATCH")
    if mode == "baseline":
        return baseline
    if mode not in {"fault", "variation"} or mutation is None:
        raise EvaluatorDefect("INVALID_REPLAY_CONFIGURATION")
    # Only the designated baseline is replaced. Other requests in the same
    # run still receive their own frozen baseline responses.
    if mutation.get("baseline_id") != baseline["baseline_id"]:
        return baseline
    if mutation.get("scenario_id") not in {None, scenario_id}:
        raise EvaluatorDefect("MUTATION_SCENARIO_LINK_MISMATCH")
    return mutation

# END_STAGE06_ROUTING_MODULE

from pathlib import Path

import importlib
import json
import time


CELL9_STARTED = time.time()

require_cell_pass(8)

routing_module_path = (
    PROJECT_ROOT
    / "src/routing.py"
)

routing_write_status = freeze_visible_module(
    "ROUTING",
    routing_module_path,
)

importlib.invalidate_caches()

import routing
if sha256_file(PROJECT_ROOT / "src/routing.py") != stage05.manifest["routing_sha256"]:
    raise RuntimeError("Evaluator routing differs from the qualified shared builder.")

if (
    routing.ROUTING_VERSION
    != ROUTING_VERSION
):
    raise RuntimeError(
        "Routing module version mismatch."
    )

cell9_report = save_cell_report(
    9,
    routing_version=ROUTING_VERSION,
    routing_module_sha256=sha256_file(
        routing_module_path
    ),
    module_integrity="HASH_VERIFIED",
    unauthorized_host_policy="REJECT",
    unknown_scenario_policy="UNSUPPORTED",
    permissive_default_response=False,
)

print(
    json.dumps(
        cell9_report,
        indent=2,
    )
)

print("\nNOTEBOOK 06 — CELL 9: PASS")

{
  "routing_version": "stage06-openapi-contract-v6",
  "routing_module_sha256": "79bce0046c73935d9eaeebf57987535db7bfc05a78490d9105db1df3d6be3898",
  "module_integrity": "HASH_VERIFIED",
  "unauthorized_host_policy": "REJECT",
  "unknown_scenario_policy": "UNSUPPORTED",
  "permissive_default_response": false,
  "status": "PASS",
  "notebook": "06",
  "cell": 9
}

NOTEBOOK 06 — CELL 9: PASS


In [10]:
# NOTEBOOK 06 — CELL 10
# Qualify the OpenAPI evaluator and independently audit the frozen oracle.
from collections import Counter
import json

import routing

require_cell_pass(9)


def qualify_success_status_mapping(routing):
    """Exercise the same status policy at the builder and evaluator boundaries."""
    scenario = {
        "scenario_id": "status-fixture", "path_template": "/items", "method": "get",
        "applicability": "APPLICABLE", "expected_validity": "VALID",
        "intent": "valid_baseline_request", "parameter_values": {},
        "documented_response_mapping": {"status": 201},
    }
    baseline = {"scenario_id": "status-fixture", "baseline_id": "status-baseline",
                "status": 201, "body": None}
    request = routing.normalize_prepared_request("GET", "http://openapi.test/items", None, None)

    def document(statuses):
        return {"openapi": "3.0.3", "paths": {"/items": {"get": {
            "responses": {code: {"description": "fixture response"} for code in statuses}
        }}}}

    unique = document(["201", "400"])
    assert routing.success_status_mapping(unique, "/items", "get") == {"status": 201}
    routing.audit_frozen_response(unique, scenario, baseline, baseline)
    assert routing.select_frozen_response(
        routing.classify_request(request, [scenario], unique), [baseline]) == baseline

    for statuses in (["201", "204"], ["204", "201"], ["201", "2XX"], ["2XX"]):
        ambiguous = document(statuses)
        for check in (
            lambda: routing.success_status_mapping(ambiguous, "/items", "get"),
            lambda: routing.classify_request(request, [scenario], ambiguous),
            lambda: routing.audit_frozen_response(ambiguous, scenario, baseline, baseline),
        ):
            try:
                check()
            except routing.EvaluatorUnavailable as error:
                if error.code != "AMBIGUOUS_SUCCESS_STATUS_MAPPING":
                    raise
            else:
                raise RuntimeError("Ambiguous success status produced a response mapping")
    assert routing.success_status_mapping(document(["default", "400"]), "/items", "get") is None
    return {"policy": routing.SUCCESS_STATUS_POLICY, "unique_explicit_status": "PASS",
            "ambiguous_statuses_blocked": "PASS", "default_only_unmapped": "PASS"}

status_qualification = qualify_success_status_mapping(routing)
if status_qualification != stage05.manifest.get("success_status_qualification"):
    raise RuntimeError("Stage 05 did not freeze the qualified success-status policy.")

def request(method, path, body=None, headers=None):
    return routing.normalize_prepared_request(method, "http://openapi.test" + path,
                                              headers, body)


def scenario(identity, path="/items/{id}", *, value=1, validity="VALID",
             intent="valid_baseline_request", mapping=200):
    return {
        "scenario_id": identity, "path_template": path, "method": "get",
        "applicability": "APPLICABLE", "expected_validity": validity,
        "parameter_values": {"path": {"id": value}}, "intent": intent,
        "documented_response_mapping": {"status": mapping} if mapping else None,
    }


def must_reject(action, error_type, reason):
    try:
        action()
    except error_type as error:
        if error.code == reason:
            return
        raise RuntimeError(f"Expected {reason}, got {error.code}") from error
    raise RuntimeError("Expected rejection: " + reason)


item_doc = {"openapi": "3.0.3", "paths": {"/items/{id}": {"get": {
    "parameters": [{"name": "id", "in": "path", "required": True,
                    "schema": {"type": "integer", "minimum": 1}}],
    "responses": {"200": {"description": "Fixture success"}}}}}}
positive = scenario("positive")
wrong_type = scenario("wrong_type", value="wrong", validity="INVALID",
                      intent="parameter_invalid_wrong_type", mapping=None)
too_small = scenario("too_small", value=0, validity="INVALID",
                     intent="parameter_invalid_below_minimum", mapping=None)
fixture_scenarios = [positive, wrong_type, too_small]
alternative = routing.classify_request(request("GET", "/items/12345"),
                                       fixture_scenarios, item_doc)
assert alternative["scenario_id"] == "positive"
assert alternative["typed_parameters"]["path"]["id"] == 12345
for url, identity in [("/items/0", "too_small"), ("/items/abc", "wrong_type")]:
    matched = routing.classify_request(request("GET", url), fixture_scenarios, item_doc)
    assert matched["scenario_id"] == identity
    must_reject(lambda: routing.select_frozen_response(matched, []),
                routing.EvaluatorUnavailable, "NO_DOCUMENTED_RESPONSE_MAPPING")
must_reject(lambda: routing.classify_request(request("GET", "/items/0"),
                                             [positive, wrong_type], item_doc),
            routing.EvaluatorUnavailable, "AMBIGUOUS_OR_UNMAPPED_REQUEST_SEMANTICS")
with_header = json.loads(json.dumps(item_doc))
with_header["paths"]["/items/{id}"]["get"]["parameters"].append({
    "in": "header", "name": "X-Required", "required": True,
    "schema": {"type": "string"},
})
must_reject(lambda: routing.classify_request(request("GET", "/items/1"),
                                             fixture_scenarios, with_header),
            routing.EvaluatorUnavailable, "AMBIGUOUS_OR_UNMAPPED_REQUEST_SEMANTICS")
assert routing.classify_request(request("GET", "/items/1", headers={"X-Required": "v"}),
                                fixture_scenarios, with_header)["scenario_id"] == "positive"
must_reject(lambda: routing.classify_request(routing.normalize_prepared_request(
    "GET", "https://openapi.test/items/1"), fixture_scenarios, item_doc),
    routing.UnsupportedRoute, "UNAUTHORIZED_HOST")
must_reject(lambda: routing.classify_request(request("POST", "/items/1"),
                                             fixture_scenarios, item_doc),
            routing.UnsupportedRoute, "REQUEST_OUTSIDE_TARGET_OPERATION")
assert routing.match_path_template("/item/{id}", "/item/a%2Fb") == {"id": "a/b"}
assert request("GET", "/items/1?q=1&q=2", headers={"Cookie": "sid=abc"})["query"] == {"q": ["1", "2"]}
assert request("GET", "/items/1", headers={"Cookie": "sid=abc"})["cookies"] == {"sid": "abc"}

post_doc = {"openapi": "3.0.3",
            "components": {"schemas": {"Age": {"type": "object", "required": ["age"],
                           "properties": {"age": {"type": "integer"}}}}},
            "paths": {"/users": {"post": {"requestBody": {"required": True, "content": {
                "application/json": {"schema": {"$ref": "#/components/schemas/Age"}},
                "application/x-www-form-urlencoded": {"schema": {"type": "object",
                    "required": ["name"], "properties": {"name": {"type": "string"}}}},
            }}, "responses": {"201": {"description": "Fixture success"}}}}}}
post_scenario = {**scenario("post", "/users", mapping=201), "method": "post",
                 "parameter_values": {}, "body": {"age": 25}}
for content_type, body in [("application/json", b'{"age":41}'),
                           ("application/x-www-form-urlencoded", b"name=A")]:
    assert routing.classify_request(request("POST", "/users", body,
        {"Content-Type": content_type}), [post_scenario], post_doc)["scenario_id"] == "post"
for body, content_type in [(b'{"name":"A"}', "application/json"),
                            (b"null", "application/json"), (b"{", "application/json")]:
    must_reject(lambda: routing.classify_request(request("POST", "/users", body,
        {"Content-Type": content_type}), [post_scenario], post_doc),
        routing.EvaluatorUnavailable, "AMBIGUOUS_OR_UNMAPPED_REQUEST_SEMANTICS")
remote_doc = json.loads(json.dumps(post_doc))
remote_doc["paths"]["/users"]["post"]["requestBody"]["content"]["application/json"]["schema"] = {
    "$ref": "https://example.invalid/schema"
}
must_reject(lambda: routing.classify_request(request("POST", "/users", b"{}",
    {"Content-Type": "application/json"}), [post_scenario], remote_doc),
    routing.EvaluatorUnavailable, "UNRESOLVED_SCHEMA_REFERENCE")
headers, payload = routing.prepare_frozen_response(
    {"body": "value"}, {"source_evidence": [{"media_type": "application/json"}]})
assert headers["Content-Type"] == "application/json" and json.loads(payload) == "value"
baseline = {"scenario_id": "positive", "baseline_id": "b1", "status": 200, "body": {}}
other_fault = {"scenario_id": "other", "baseline_id": "b2", "status": 200, "body": []}
assert routing.select_frozen_response(alternative, [baseline], "fault", other_fault) == baseline
own_fault = {**other_fault, "baseline_id": "b1", "scenario_id": "positive"}
assert routing.select_frozen_response(alternative, [baseline], "fault", own_fault) == own_fault
must_reject(lambda: routing.select_frozen_response(alternative, []),
            routing.EvaluatorDefect, "FROZEN_BASELINE_LINK_COUNT")

counts = Counter()
blockers = []
unmapped = []
serialization_rules = []
for selected in selection:
    op_id = selected["op_id"]
    root = PROJECT_ROOT / "oracles/frozen/pilot" / op_id
    canonical_path = PROJECT_ROOT / "data/inputs/L0" / f"{op_id}.json"
    document = read_json(canonical_path)
    scenarios = read_jsonl_file(root / "scenarios.jsonl")
    baselines = read_jsonl_file(root / "baselines.jsonl")
    mutations = {
        "fault": read_jsonl_file(root / "response_faults.jsonl"),
        "variation": read_jsonl_file(root / "legitimate_variations.jsonl"),
    }
    supplements = root / "response_fault_supplements.jsonl"
    if supplements.is_file():
        mutations["fault"].extend(read_jsonl_file(supplements))
    scenario_by_id = {s["scenario_id"]: s for s in scenarios}
    baseline_by_id = {b["baseline_id"]: b for b in baselines}
    if len(scenario_by_id) != len(scenarios) or len(baseline_by_id) != len(baselines):
        raise RuntimeError(f"Duplicate frozen oracle identity: {op_id}")
    counts["operations"] += 1
    for s in scenarios:
        # Compare with the applicable scenario denominator from Cell 5.
        counts["scenarios"] += int(s.get("applicability") == "APPLICABLE")
        if s.get("documented_response_mapping") is None:
            unmapped.append({"op_id": op_id, "scenario_id": s["scenario_id"],
                             "intent": s["intent"], "reason": "NO_DOCUMENTED_RESPONSE_MAPPING"})
        elif sum(b["scenario_id"] == s["scenario_id"] for b in baselines) != 1:
            blockers.append({"op_id": op_id, "scenario_id": s["scenario_id"],
                             "reason": "MAPPED_SCENARIO_WITHOUT_UNIQUE_BASELINE"})
    for b in baselines:
        s = scenario_by_id[b["scenario_id"]]
        counts["registered_baselines"] += 1
        try:
            routing.audit_frozen_response(document, s, b, b)
        except routing.UnsupportedRoute as error:
            blockers.append({"op_id": op_id, "baseline_id": b["baseline_id"],
                             "reason": error.code, "detail": error.detail})
        else:
            counts["spec_conforming_baseline_responses"] += 1
        try:
            prepared = routing.prepare_frozen_fixture(document, s)
            classification = routing.classify_request(prepared, scenarios, document)
            chosen = routing.select_frozen_response(classification, baselines)
            if chosen["baseline_id"] != b["baseline_id"]:
                raise routing.EvaluatorDefect("CANONICAL_FIXTURE_MISROUTED")
            routing.prepare_frozen_response(chosen, b)
        except routing.UnsupportedRoute as error:
            blockers.append({"op_id": op_id, "scenario_id": s["scenario_id"],
                             "reason": error.code, "detail": error.detail})
        else:
            counts["qualified_baselines"] += 1
            rule = routing.legacy_form_rule(document, s["path_template"], s["method"])
            if rule:
                assert rule["op_id"] == op_id
                assert prepared["media_type"] == rule["wire_media_type"]
                assert prepared["body"] == {"phrase": "A", "context[]": "home"}
                assert classification["observed_violations"] == []
                multiple_contexts = {
                    **prepared, "body": {**prepared["body"], "context[]": ["home", "public"]},
                }
                assert routing.classify_request(
                    multiple_contexts, scenarios, document)["scenario_id"] == s["scenario_id"]
                unbracketed = {**prepared, "body": {"phrase": "A", "context": "home"}}
                must_reject(lambda: routing.classify_request(unbracketed, scenarios, document),
                            routing.EvaluatorUnavailable, "UNSUPPORTED_UNBRACKETED_FORM_ARRAY")
                wrong_media = {**prepared, "media_type": rule["declared_media_type"]}
                must_reject(lambda: routing.classify_request(wrong_media, scenarios, document),
                            routing.EvaluatorUnavailable, "UNSUPPORTED_LEGACY_FORM_WIRE_TYPE")
                missing_phrase = {**prepared, "body": {"context[]": "home"}}
                negative = routing.classify_request(missing_phrase, scenarios, document)
                assert negative["positive"] is False
                must_reject(lambda: routing.select_frozen_response(negative, baselines),
                            routing.EvaluatorUnavailable, "NO_DOCUMENTED_RESPONSE_MAPPING")
                serialization_rules.append({
                    **rule, "scenario_id": s["scenario_id"], "baseline_id": b["baseline_id"],
                    "request_roundtrip_verified": True,
                })
    for kind, records in mutations.items():
        for row in records:
            counts[f"registered_{kind}s"] += 1
            baseline = baseline_by_id.get(row["baseline_id"])
            if baseline is None or row.get("scenario_id") not in {None, baseline["scenario_id"]}:
                blockers.append({"op_id": op_id, "kind": kind,
                                 "reason": "ORPHAN_OR_MISLINKED_REPLAY_RECORD",
                                 "record_id": row.get("fault_id") or row.get("variation_id")})
                continue
            linked = scenario_by_id[baseline["scenario_id"]]
            classification = {
                "scenario_id": linked["scenario_id"],
                "documented_response_mapping": linked["documented_response_mapping"],
            }
            if routing.select_frozen_response(classification, baselines, kind, row) is not row:
                raise RuntimeError(f"Frozen {kind} selected wrong response: {op_id}")
            routing.prepare_frozen_response(row, baseline)
            counts[f"linked_{kind}s"] += 1
            if kind == "variation":
                try:
                    routing.audit_frozen_response(document, linked, row, baseline)
                except routing.UnsupportedRoute as error:
                    blockers.append({"op_id": op_id, "variation_id": row["variation_id"],
                                     "reason": error.code, "detail": error.detail})
                else:
                    counts["spec_conforming_variation_responses"] += 1
            else:
                try:
                    routing.audit_frozen_response(document, linked, row, baseline)
                except routing.EvaluatorDefect as error:
                    if error.code not in {"FROZEN_RESPONSE_SCHEMA_MISMATCH",
                                          "UNDOCUMENTED_FROZEN_RESPONSE_BODY",
                                          "MISSING_REQUIRED_FROZEN_RESPONSE_HEADER",
                                          "UNDOCUMENTED_FROZEN_RESPONSE_STATUS"}:
                        blockers.append({"op_id": op_id, "fault_id": row["fault_id"],
                                         "reason": error.code, "detail": error.detail})
                    else:
                        counts["spec_violating_fault_responses"] += 1
                except routing.EvaluatorUnavailable as error:
                    blockers.append({"op_id": op_id, "fault_id": row["fault_id"],
                                     "reason": error.code, "detail": error.detail})
                else:
                    blockers.append({"op_id": op_id, "fault_id": row["fault_id"],
                                     "reason": "REGISTERED_FAULT_CONFORMS_TO_SPEC"})

denominators = read_jsonl_file(PROJECT_ROOT / "data/audits/cell05_frozen_denominators.jsonl")
assert len(denominators) == counts["operations"] == 100
for audit_key, denominator_key in (
    ("scenarios", "scenarios"),
    ("registered_baselines", "baselines"),
    ("registered_faults", "faults"),
    ("registered_variations", "variations"),
):
    expected = sum(record[denominator_key] for record in denominators)
    if counts[audit_key] != expected:
        raise RuntimeError(f"Frozen oracle inventory mismatch: {audit_key}")
status = "BLOCKED" if blockers else "PASS"
audit = {"status": status, "counts": dict(counts), "blocking_findings": blockers,
         "unmapped_negative_scenarios": unmapped, "routing_version": routing.ROUTING_VERSION,
         "request_serialization_rules": serialization_rules,
         "stage05_manifest_sha256": STAGE_ARCHIVES["05"].internal_manifest_sha256}
audit_path = PROJECT_ROOT / "reports/qualification/canonical_oracle_audit.json"
write_json_immutable(audit_path, audit)
print(json.dumps({"status": status, "counts": dict(counts),
                  "unmapped_scenario_count": len(unmapped), "blockers": blockers}, indent=2))
if blockers:
    raise RuntimeError(
        f"CHECKPOINT_B_REVALIDATION_BLOCK: {len(blockers)} frozen oracle issue(s). "
        f"See {audit_path}. Resolve from upstream specification evidence before qualification."
    )

write_json_immutable(PROJECT_ROOT / "reports/qualification/routing_fixtures.json", {
    "status": "PASS", "routing_version": routing.ROUTING_VERSION,
    "fixture_groups": ["typed_values", "negative_attribution", "required_parameters",
                       "serialization", "media_selection", "references", "response_selection"],
    "success_status_qualification": status_qualification,
})
write_json_immutable(PROJECT_ROOT / "reports/qualification/real_oracle_routing_preflight.json",
                     {"status": "PASS", "counts": dict(counts),
                      "oracle_audit_sha256": sha256_file(audit_path)})
cell10_report = save_cell_report(10, qualification_status="PASS",
                                frozen_oracle_counts=dict(counts))
print(json.dumps(cell10_report, indent=2))
print("\nNOTEBOOK 06 — CELL 10: PASS")


{
  "status": "PASS",
  "counts": {
    "operations": 100,
    "scenarios": 1007,
    "registered_baselines": 114,
    "spec_conforming_baseline_responses": 114,
    "qualified_baselines": 114,
    "registered_faults": 733,
    "linked_faults": 733,
    "spec_violating_fault_responses": 733,
    "registered_variations": 158,
    "linked_variations": 158,
    "spec_conforming_variation_responses": 158
  },
  "unmapped_scenario_count": 904,
  "blockers": []
}
{
  "qualification_status": "PASS",
  "frozen_oracle_counts": {
    "operations": 100,
    "scenarios": 1007,
    "registered_baselines": 114,
    "spec_conforming_baseline_responses": 114,
    "qualified_baselines": 114,
    "registered_faults": 733,
    "linked_faults": 733,
    "spec_violating_fault_responses": 733,
    "registered_variations": 158,
    "linked_variations": 158,
    "spec_conforming_variation_responses": 158
  },
  "status": "PASS",
  "notebook": "06",
  "cell": 10
}

NOTEBOOK 06 — CELL 10: PASS


In [11]:
# NOTEBOOK 06 — CELL 11
# Implement isolated pytest collection and execution.
# No bare-subprocess fallback is permitted.

# BEGIN_STAGE06_WORKER_MODULE

from __future__ import annotations

from pathlib import Path
import base64
import ctypes.util
import importlib.metadata
import re
import sysconfig
import json
import os
import platform
import resource
import selectors
import shutil
import signal
import socket
import socketserver
import types
import subprocess
import sys
import tempfile
import threading
import time

from routing import (
    normalize_prepared_request,
    classify_request,
    select_frozen_response,
    prepare_frozen_response,
    UnsupportedRoute,
)


WORKER_VERSION = "stage06-isolated-worker-v11-single-opportunity"
REPLAY_TARGET_POLICY = "first_delivered_request_in_passing_item"


# This is the trusted entry script executed INSIDE the sandbox.
# It is deliberately separate from generated test source.
SANDBOX_BOOTSTRAP = r'''
import base64
import ctypes
import ctypes.util
import importlib.metadata
import re
import sysconfig
import errno
import json
import os
import socket
import sys
import sysconfig

# Python -S prevents all host/user site hooks. Add only the packaged site root.
site_path = sysconfig.get_path("purelib")
version_dir = "python%d.%d" % sys.version_info[:2]
for path in (site_path, sysconfig.get_path("platlib"),
             "/usr/local/lib/" + version_dir + "/dist-packages",
             "/usr/local/lib/" + version_dir + "/site-packages",
             "/usr/lib/" + version_dir + "/dist-packages",
             "/usr/lib/python3/dist-packages"):
    if path not in sys.path:
        sys.path.append(path)

libname = "libseccomp.so.2"

if not libname:
    raise SystemExit("SECURITY_BLOCK: libseccomp missing")

lib = ctypes.CDLL(libname, use_errno=True)

libc = ctypes.CDLL(None, use_errno=True)
PR_SET_NO_NEW_PRIVS = 38
if libc.prctl(PR_SET_NO_NEW_PRIVS, 1, 0, 0, 0) != 0:
    raise SystemExit("SECURITY_BLOCK: PR_SET_NO_NEW_PRIVS failed")

lib.seccomp_init.argtypes = [ctypes.c_uint32]
lib.seccomp_init.restype = ctypes.c_void_p

lib.seccomp_syscall_resolve_name.argtypes = [
    ctypes.c_char_p
]
lib.seccomp_syscall_resolve_name.restype = ctypes.c_int

lib.seccomp_rule_add.argtypes = [
    ctypes.c_void_p,
    ctypes.c_uint32,
    ctypes.c_int,
    ctypes.c_uint,
]
lib.seccomp_rule_add.restype = ctypes.c_int

class ScmpArgCmp(ctypes.Structure):
    _fields_ = [("arg", ctypes.c_uint), ("op", ctypes.c_uint32),
                ("datum_a", ctypes.c_uint64), ("datum_b", ctypes.c_uint64)]

lib.seccomp_rule_add_array.argtypes = [ctypes.c_void_p, ctypes.c_uint32,
                                        ctypes.c_int, ctypes.c_uint,
                                        ctypes.POINTER(ScmpArgCmp)]
lib.seccomp_rule_add_array.restype = ctypes.c_int

lib.seccomp_load.argtypes = [ctypes.c_void_p]
lib.seccomp_load.restype = ctypes.c_int

lib.seccomp_release.argtypes = [ctypes.c_void_p]
lib.seccomp_release.restype = None

SCMP_ACT_ALLOW = 0x7FFF0000
SCMP_ACT_ERRNO = 0x00050000 | errno.EPERM

context = lib.seccomp_init(SCMP_ACT_ALLOW)

if not context:
    raise SystemExit("SECURITY_BLOCK: seccomp_init failed")

try:
    blocked_syscalls = (
        b"execve",
        b"execveat",
        b"fork",
        b"vfork",
        b"clone",
        b"clone3",
        b"ptrace",
        b"socketpair",
        b"connect",
        b"bind",
        b"listen",
        b"accept",
        b"accept4",
        b"io_uring_setup",
        b"io_uring_enter",
        b"io_uring_register",
        b"pidfd_getfd",
        b"mount",
        b"umount2",
        b"pivot_root",
        b"chroot",
        b"unshare",
        b"setns",
        b"bpf",
        b"perf_event_open",
        b"process_vm_readv",
        b"process_vm_writev",
        b"keyctl",
        b"add_key",
        b"request_key",
    )

    for name in blocked_syscalls:
        number = lib.seccomp_syscall_resolve_name(name)

        if number < 0:
            continue

        result = lib.seccomp_rule_add(
            context,
            SCMP_ACT_ERRNO,
            number,
            0,
        )

        if result != 0:
            raise RuntimeError(
                "Cannot deny syscall: "
                + name.decode()
            )

    # Deny ALL new sockets, including abstract Unix sockets.
    # The trusted parent hands off one connected socketpair descriptor.
    socket_syscall = lib.seccomp_syscall_resolve_name(b"socket")
    if socket_syscall < 0:
        raise RuntimeError("socket syscall not resolvable")
    if lib.seccomp_rule_add(context, SCMP_ACT_ERRNO,
                             socket_syscall, 0) != 0:
        raise RuntimeError("Cannot prohibit socket creation")

    if lib.seccomp_load(context) != 0:
        raise RuntimeError("seccomp_load failed")

finally:
    lib.seccomp_release(context)


import requests
from requests.adapters import BaseAdapter

offline_router_connection = socket.socket(
    fileno=int(os.environ["STAGE06_ROUTER_FD"])
)
offline_router_connection.settimeout(3)
current_item_id = None


class OfflineAdapter(BaseAdapter):
    def send(self, request, **kwargs):
        body = request.body

        if isinstance(body, str):
            body = body.encode("utf-8")

        payload = {
            "method": request.method,
            "pytest_item_id": current_item_id,
            "url": request.url,
            "headers": dict(request.headers),
            "body_b64": base64.b64encode(
                body or b""
            ).decode("ascii"),
        }

        # Connected descriptor was created by the trusted parent before
        # installing seccomp. Generated code cannot create/connect sockets.
        connection = offline_router_connection

        connection.sendall(
            json.dumps(payload).encode("utf-8")
            + b"\n"
        )

        response_bytes = b""

        while b"\n" not in response_bytes:
            chunk = connection.recv(65536)

            if not chunk:
                raise RuntimeError(
                    "Offline coordinator disconnected"
                )

            response_bytes += chunk

            if len(response_bytes) > 2_000_000:
                raise RuntimeError(
                    "Offline coordinator reply too large"
                )

        result = json.loads(
            response_bytes.split(b"\n", 1)[0]
        )

        if result.get("error"):
            raise RuntimeError(
                "EVALUATOR_UNSUPPORTED_ROUTE:"
                + result["error"]
            )

        response = requests.Response()

        response.status_code = int(
            result["status"]
        )

        response._content = base64.b64decode(
            result["body_b64"]
        )

        response.headers = (
            requests.structures.CaseInsensitiveDict(
                result.get("headers") or {}
            )
        )

        response.url = request.url
        response.request = request

        return response

    def close(self):
        pass


original_session_init = requests.Session.__init__


def offline_session_init(
    self,
    *args,
    **kwargs,
):
    original_session_init(
        self,
        *args,
        **kwargs,
    )

    self.mount(
        "http://",
        OfflineAdapter(),
    )

    self.mount(
        "https://",
        OfflineAdapter(),
    )


requests.Session.__init__ = offline_session_init


import pytest

pytest_arguments = [
    "-q",
    "--capture=no",
    "-p",
    "no:cacheprovider",
    "--disable-warnings",
    "--tb=short",
]

if os.environ.get("STAGE06_PHASE") == "collect":
    pytest_arguments.append("--collect-only")

pytest_arguments.append(
    "/work/generated_test.py"
)

class OutcomeRecorder:
    def __init__(self):
        self.outcomes = {}
        self.assertion_failures = set()

    @pytest.hookimpl(hookwrapper=True)
    def pytest_runtest_protocol(self, item, nextitem):
        global current_item_id
        current_item_id = item.nodeid
        try:
            yield
        finally:
            current_item_id = None

    @pytest.hookimpl(hookwrapper=True)
    def pytest_runtest_makereport(self, item, call):
        result = yield
        report = result.get_result()
        if (report.when == "call" and report.failed and not hasattr(report, "wasxfail")
                and call.excinfo is not None and call.excinfo.errisinstance(AssertionError)):
            self.assertion_failures.add(item.nodeid)

    def pytest_runtest_logreport(self, report):
        node_id = report.nodeid
        if report.failed:
            self.outcomes[node_id] = "failed" if report.when == "call" else "error"
        elif report.skipped and node_id not in self.outcomes:
            self.outcomes[node_id] = "xfailed" if hasattr(report, "wasxfail") else "skipped"
        elif report.when == "call" and report.passed:
            self.outcomes[node_id] = "xpassed" if hasattr(report, "wasxfail") else "passed"


recorder = OutcomeRecorder()
print("STAGE06_PYTEST_READY", flush=True)
exit_code = pytest.main(pytest_arguments, plugins=[recorder])
if os.environ.get("STAGE06_PHASE") != "collect":
    print("\nSTAGE06_ASSERTION_FAILURES=" + json.dumps(sorted(recorder.assertion_failures)), flush=True)
    print("\nSTAGE06_ITEM_OUTCOMES=" + json.dumps(recorder.outcomes, sort_keys=True), flush=True)

sys.exit(
    int(exit_code)
)
'''


def enforce_resource_limits():
    limits = (
        (resource.RLIMIT_CPU, 20),
        (resource.RLIMIT_AS, 2 * 1024**3),
        (resource.RLIMIT_FSIZE, 2 * 1024**2),
        (resource.RLIMIT_NOFILE, 64),
        (resource.RLIMIT_NPROC, 32),
    )

    for resource_kind, soft_limit in limits:
        current_soft, current_hard = (
            resource.getrlimit(resource_kind)
        )

        actual_soft = (
            soft_limit
            if current_hard < 0
            else min(
                soft_limit,
                current_hard,
            )
        )

        resource.setrlimit(
            resource_kind,
            (
                actual_soft,
                current_hard,
            ),
        )



# The runtime template contains only Python, its bounded standard library,
# pytest/requests dependencies and required ELF libraries; it NEVER contains
# /kaggle/input, the project tree, environment secrets or hidden oracle files.
_RUNTIME_TEMPLATE = None
_BOUNDARY_QUALIFIED_REPORT = None
_CHROOT_COMMAND = shutil.which("chroot")


def _copy_host_file(source_path, root):
    p = Path(source_path)
    if not p.is_absolute() or not p.exists() or p.is_dir():
        raise RuntimeError("SECURITY_BLOCK: invalid runtime source " + str(p))
    destination = root / str(p).lstrip("/")
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists():
        shutil.copy2(p, destination, follow_symlinks=True)


def _copy_dynamic_dependencies(source_paths, root):
    """Only trusted ELF libraries referenced by the interpreter/extensions."""
    paths = [Path(p) for p in source_paths]
    copied = set()
    while paths:
        src = paths.pop()
        if str(src) in copied or not src.is_file():
            continue
        copied.add(str(src))
        probe = subprocess.run(["ldd", str(src)], stdout=subprocess.PIPE,
                               stderr=subprocess.PIPE, text=True, timeout=15,
                               check=False)
        if probe.returncode != 0:
            # Some Python extension .so files are static/no dynamic deps.
            if "not a dynamic executable" in probe.stderr.lower():
                continue
            raise RuntimeError("SECURITY_BLOCK: ELF dependency probe failed: "
                               + str(src) + ": " + probe.stderr[:300])
        for dependency in re.findall(r"(?<![A-Za-z0-9_])(/[^\s()]+)",
                                     probe.stdout):
            dep = Path(dependency)
            if not dep.is_file():
                raise RuntimeError("SECURITY_BLOCK: missing shared library " + dependency)
            _copy_host_file(dep, root)
            if str(dep) not in copied:
                paths.append(dep)


def _copy_package_distributions(root):
    # Frozen minimal import surface required by generated requests/pytest tests.
    names = (
        "pytest", "requests", "urllib3", "idna", "charset-normalizer",
        "certifi", "pluggy", "iniconfig", "packaging", "colorama",
        "typing-extensions", "tomli", "exceptiongroup",
        "importlib-metadata", "zipp", "attrs", "pygments",
        "jsonschema", "jsonschema-specifications", "referencing", "rpds-py",
        "six", "PyYAML", "pydantic", "pydantic-core",
    )
    required = {"pytest", "requests", "urllib3", "idna", "certifi",
                "pluggy", "iniconfig", "packaging"}
    copied_shared_objects = set()
    found = set()
    for name in names:
        try:
            dist = importlib.metadata.distribution(name)
        except importlib.metadata.PackageNotFoundError:
            if name in required:
                raise RuntimeError("SECURITY_BLOCK: dependency missing: " + name)
            continue
        found.add(name)
        if not dist.files:
            raise RuntimeError("SECURITY_BLOCK: missing distribution file list " + name)
        for relative in dist.files:
            src = Path(dist.locate_file(relative))
            if not src.is_file():
                continue
            # Only the registered distribution and metadata, never arbitrary
            # user/project paths or unrelated site packages.
            if not any(part in src.parts for part in ("site-packages", "dist-packages")):
                continue
            _copy_host_file(src, root)
            if ".so" in src.name:
                copied_shared_objects.add(src)
    return copied_shared_objects


def _build_runtime_template():
    global _RUNTIME_TEMPLATE
    if _RUNTIME_TEMPLATE is not None and _RUNTIME_TEMPLATE.is_dir():
        return _RUNTIME_TEMPLATE
    if platform.system() != "Linux" or os.geteuid() != 0 or not _CHROOT_COMMAND:
        raise RuntimeError(
            "SECURITY_BLOCK: Kaggle OS worker requires root with CAP_SYS_CHROOT "
            "and the system chroot utility; no unisolated fallback is permitted"
        )
    if not ctypes.util.find_library("seccomp"):
        raise RuntimeError("SECURITY_BLOCK: libseccomp unavailable")
    runtime_dir = Path(tempfile.mkdtemp(prefix="stage06_osruntime_"))
    try:
        python_exe = Path(sys.executable)
        if not python_exe.is_file():
            raise RuntimeError("SECURITY_BLOCK: Python executable not found")
        stdlib = Path(sysconfig.get_path("stdlib"))
        if not stdlib.is_dir():
            raise RuntimeError("SECURITY_BLOCK: Python stdlib missing")
        _copy_host_file(python_exe, runtime_dir)
        target_stdlib = runtime_dir / str(stdlib).lstrip("/")

        def ignore(directory, files):
            excluded = {"site-packages", "dist-packages", "__pycache__",
                        "test", "tests", "tkinter", "idlelib", "ensurepip",
                        "venv", "turtledemo"}
            return [name for name in files if name in excluded]

        shutil.copytree(stdlib, target_stdlib, ignore=ignore,
                        symlinks=False, dirs_exist_ok=True)
        extensions = set(target_stdlib.rglob("*.so"))
        extensions.update(_copy_package_distributions(runtime_dir))

        # Host loader needs its referenced interpreter/extension libraries.
        host_extensions = [python_exe]
        for extension in extensions:
            # stdlib extensions are already copied beneath runtime_dir;
            # distribution extensions are still absolute HOST source paths.
            # Never call relative_to(runtime_dir) on a host path.
            host = (
                Path("/") / extension.relative_to(runtime_dir)
                if extension.is_relative_to(runtime_dir)
                else extension
            )
            if host.is_file():
                host_extensions.append(host)
        _copy_dynamic_dependencies(host_extensions, runtime_dir)

        # libseccomp is linked dynamically by ctypes in bootstrap.py.
        lib_candidates = [
            p for prefix in ("/lib", "/usr/lib", "/lib64", "/usr/local/lib")
            for p in Path(prefix).glob("**/libseccomp.so.2")
            if p.is_file()
        ]
        if not lib_candidates:
            raise RuntimeError("SECURITY_BLOCK: libseccomp.so.2 not found")
        seccomp_source = sorted(lib_candidates, key=lambda p: len(str(p)))[0]
        _copy_host_file(seccomp_source, runtime_dir)
        _copy_dynamic_dependencies([seccomp_source], runtime_dir)

        # Dynamic linker configuration is non-secret, needed for libpython.
        if Path("/etc/ld.so.cache").is_file():
            _copy_host_file(Path("/etc/ld.so.cache"), runtime_dir)
        # GNU chroot may resolve supplemental groups after entering the root.
        etc = runtime_dir / "etc"
        etc.mkdir(parents=True, exist_ok=True)
        (etc / "passwd").write_text(
            "root:x:0:0:root:/root:/bin/sh\n"
            "nobody:x:65534:65534:nobody:/work:/usr/sbin/nologin\n",
            encoding="utf-8",
        )
        (etc / "group").write_text(
            "root:x:0:\n"
            "nogroup:x:65534:\n",
            encoding="utf-8",
        )
        for name in ("work", "tmp", "dev"):
            (runtime_dir / name).mkdir(exist_ok=True)
        (runtime_dir / "tmp").chmod(0o1777)
        # Some pytest startup paths open os.devnull even when output
        # capture is disabled. A sandbox-local regular file is sufficient
        # and avoids exposing the host /dev tree inside the chroot.
        sandbox_devnull = runtime_dir / "dev/null"
        sandbox_devnull.write_bytes(b"")
        sandbox_devnull.chmod(0o666)
        # mkdtemp starts 0700. Without this, UID 65534 cannot traverse '/'!
        runtime_dir.chmod(0o755)
        _RUNTIME_TEMPLATE = runtime_dir
    except BaseException:
        shutil.rmtree(runtime_dir, ignore_errors=True)
        raise
    return _RUNTIME_TEMPLATE


def _create_worker_root(workspace):
    runtime = _build_runtime_template()
    worker_root = Path(workspace) / "sandbox"
    try:
        shutil.copytree(runtime, worker_root, copy_function=os.link,
                        symlinks=False)
    except OSError as error:
        raise RuntimeError(
            "SECURITY_BLOCK: cannot create fresh immutable sandbox runtime: "
            + str(error)
        ) from error
    # The synthetic dev/null is the only writable runtime file. Give each
    # worker its own inode so generated code cannot persist data across runs.
    sandbox_devnull = worker_root / "dev/null"
    sandbox_devnull.unlink()
    sandbox_devnull.touch(mode=0o666)
    sandbox_devnull.chmod(0o666)
    work = worker_root / "work"
    worker_root.chmod(0o755)
    work.chmod(0o1777)
    return worker_root, work


def construct_sandbox_command(worker_root, python_executable):
    _build_runtime_template()
    executable = Path(python_executable)
    if not (Path(worker_root) / str(executable).lstrip("/")).is_file():
        raise RuntimeError("SECURITY_BLOCK: Python missing in chroot runtime")
    return [
        _CHROOT_COMMAND,
        "--userspec=65534:65534",
        str(worker_root),
    ]


def qualify_os_boundary():
    # This is a real root-change and drop-UID test. A successful 'which chroot'
    # alone is NOT an isolation qualification.
    global _BOUNDARY_QUALIFIED_REPORT
    if _BOUNDARY_QUALIFIED_REPORT is not None:
        return dict(_BOUNDARY_QUALIFIED_REPORT)
    _build_runtime_template()
    with tempfile.TemporaryDirectory(prefix="stage06_boundary_") as directory:
        root, work = _create_worker_root(directory)
        command = construct_sandbox_command(root, sys.executable)
        command.extend([sys.executable, "-I", "-B", "-S", "-c",
                        "import os; print('BOUNDARY_OK', os.getuid(), "
                        "os.getgid(), ','.join(map(str,os.getgroups())))"])
        try:
            result = subprocess.run(command, stdout=subprocess.PIPE,
                                    stderr=subprocess.PIPE, timeout=15,
                                    check=False)
        except (subprocess.TimeoutExpired, OSError) as error:
            raise RuntimeError("SECURITY_BLOCK: chroot qualification failed") from error
        words = result.stdout.decode(errors="replace").strip().split()
        group_tokens = words[3].split(",") if len(words) > 3 else []
        if (result.returncode != 0 or len(words) < 3
                or words[:3] != ["BOUNDARY_OK", "65534", "65534"]
                or "0" in group_tokens):
            raise RuntimeError(
                "SECURITY_BLOCK: Kaggle denied an isolated chroot worker "
                "or runtime imports: " + result.stderr[-1000:].decode(errors="replace")
            )
    _BOUNDARY_QUALIFIED_REPORT = {
        "status": "PASS",
        "worker_version": WORKER_VERSION,
        "os_boundary": "chroot_with_dropped_uid_gid",
        "user_namespaces_required": False,
        "syscall_policy": "libseccomp_non_unix_network_and_process_denied",
        "resource_limits": True,
    }
    return dict(_BOUNDARY_QUALIFIED_REPORT)


def bounded_process(
    command,
    timeout,
    maximum_output,
    phase="run",
    router_fd=None,
):
    process = subprocess.Popen(
        command,
        stdin=subprocess.DEVNULL,
        pass_fds=(router_fd,),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        env={
            "PATH": "/usr/bin:/bin",
            "HOME": "/work",
            "TMPDIR": "/tmp",
            "PYTHONDONTWRITEBYTECODE": "1",
            "PYTHONNOUSERSITE": "1",
            "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1",
            "STAGE06_PHASE": phase,
            "STAGE06_ROUTER_FD": str(router_fd),
        },
        preexec_fn=enforce_resource_limits,
        start_new_session=True,
    )

    selector = selectors.DefaultSelector()

    assert process.stdout is not None
    assert process.stderr is not None

    buffers = {
        process.stdout: bytearray(),
        process.stderr: bytearray(),
    }

    for stream in buffers:
        os.set_blocking(
            stream.fileno(),
            False,
        )

        selector.register(
            stream,
            selectors.EVENT_READ,
        )

    deadline = (
        time.monotonic()
        + timeout
    )

    timed_out = False
    output_truncated = False

    try:
        while (
            selector.get_map()
            or process.poll() is None
        ):
            if time.monotonic() > deadline:
                timed_out = True
                break

            remaining_time = max(
                0.01,
                deadline - time.monotonic(),
            )

            events = selector.select(
                timeout=min(
                    0.1,
                    remaining_time,
                )
            )

            for key, _ in events:
                stream = key.fileobj

                try:
                    chunk = os.read(
                        stream.fileno(),
                        65536,
                    )

                except BlockingIOError:
                    continue

                if not chunk:
                    selector.unregister(
                        stream
                    )
                    continue

                remaining_bytes = (
                    maximum_output
                    - sum(
                        len(buffer)
                        for buffer in buffers.values()
                    )
                )

                if len(chunk) > remaining_bytes:
                    if remaining_bytes > 0:
                        buffers[stream].extend(
                            chunk[:remaining_bytes]
                        )

                    output_truncated = True
                    break

                buffers[stream].extend(
                    chunk
                )

            if output_truncated:
                break

        if timed_out or output_truncated:
            try:
                os.killpg(
                    process.pid,
                    signal.SIGKILL,
                )

            except ProcessLookupError:
                pass

        exit_code = process.wait(
            timeout=5
        )

    finally:
        selector.close()

        for stream in buffers:
            stream.close()

    return {
        "exit_code": exit_code,
        "timed_out": timed_out,
        "output_truncated": output_truncated,
        "stdout": bytes(
            buffers[process.stdout]
        ).decode(errors="replace"),
        "stderr": bytes(
            buffers[process.stderr]
        ).decode(errors="replace"),
    }


def matches_replay_target(attempt, target):
    return (
        attempt["pytest_item_id"] == target["pytest_item_id"]
        and attempt["item_request_ordinal"] == target["item_request_ordinal"]
        and attempt["baseline_id"] == target["baseline_id"]
        and attempt["scenario_id"] == target["scenario_id"]
        and json.dumps(attempt["request"], sort_keys=True)
        == json.dumps(target["request"], sort_keys=True)
    )


def make_router_coordinator(route, phase):
    mode = route.get("mode", "baseline")
    mutation = route.get("mutation")
    target = route.get("replay_target")
    if mode == "baseline":
        if mutation is not None or target is not None:
            raise ValueError("Baseline execution cannot specify a replay target or mutation")
    else:
        if mode not in {"fault", "variation"} or phase != "run":
            raise ValueError("Fault and variation replay require the run phase")
        if not isinstance(mutation, dict) or not mutation.get(f"{mode}_id"):
            raise ValueError("Replay requires an identified frozen mutation")
        if (
            not isinstance(target, dict)
            or not isinstance(target.get("pytest_item_id"), str)
            or not target["pytest_item_id"]
            or type(target.get("item_request_ordinal")) is not int
            or target["item_request_ordinal"] < 1
            or not isinstance(target.get("request"), dict)
            or not target.get("scenario_id")
            or not target.get("baseline_id")
            or target["baseline_id"] != mutation.get("baseline_id")
            or mutation.get("scenario_id") not in {None, target["scenario_id"]}
            or not any(row["baseline_id"] == target["baseline_id"]
                       and row["scenario_id"] == target["scenario_id"]
                       for row in route["baselines"])
        ):
            raise ValueError("Replay requires a request occurrence linked to its frozen baseline")

    return types.SimpleNamespace(
        scenarios=route["scenarios"],
        operation_document=route["operation_document"],
        baselines=route["baselines"],
        mode=mode,
        mutation=mutation,
        replay_target=target,
        mutation_applied=False,
        item_request_ordinals={},
        request_log=[],
    )


class OfflineRouterHandler(
    socketserver.StreamRequestHandler
):
    def handle(self):
        while self.handle_one():
            pass

    def handle_one(self):
        coordinator = self.server

        request_line = self.rfile.readline(2_000_000)
        if not request_line:
            return False

        original = None
        prepared = None
        classification = None
        item_id = None
        item_ordinal = None
        try:
            original = json.loads(request_line)
            item_id = original.get("pytest_item_id")
            item_ordinal = coordinator.item_request_ordinals.get(item_id, 0) + 1
            coordinator.item_request_ordinals[item_id] = item_ordinal
            body = base64.b64decode(original["body_b64"], validate=True)
            prepared = normalize_prepared_request(
                original["method"], original["url"], original.get("headers"), body,
            )
            classification = classify_request(
                prepared, coordinator.scenarios, coordinator.operation_document,
            )
            baseline = select_frozen_response(classification, coordinator.baselines)
            inject = (
                coordinator.replay_target is not None
                and not coordinator.mutation_applied
                and matches_replay_target({
                    "pytest_item_id": item_id,
                    "item_request_ordinal": item_ordinal,
                    "baseline_id": baseline["baseline_id"],
                    "scenario_id": classification["scenario_id"],
                    "request": prepared,
                }, coordinator.replay_target)
            )
            selected = (select_frozen_response(
                classification, coordinator.baselines,
                mode=coordinator.mode, mutation=coordinator.mutation,
            ) if inject else baseline)
            headers, body_bytes = prepare_frozen_response(selected, baseline)
            response = {
                "status": int(selected["status"]), "headers": headers,
                "body_b64": base64.b64encode(body_bytes).decode("ascii"),
            }
            evidence = {"status": "DELIVERED", "baseline_id": baseline["baseline_id"],
                        "response_status": response["status"],
                        "replay_mutation_applied": inject,
                        "fault_id": selected.get("fault_id") if inject else None,
                        "variation_id": selected.get("variation_id") if inject else None}
        except UnsupportedRoute as error:
            response = {"error": error.code}
            evidence = {"status": "UNSUPPORTED", "diagnostic": str(error),
                        "reason_code": error.code, "failure_owner": error.failure_owner,
                        "detail": error.detail}
        except Exception as error:
            response = {"error": "OFFLINE_EVALUATOR_INTERNAL_ERROR"}
            evidence = {"status": "UNSUPPORTED", "diagnostic": str(error),
                        "reason_code": "OFFLINE_EVALUATOR_INTERNAL_ERROR",
                        "failure_owner": "EVALUATOR_FAILURE"}
        self.wfile.write(
            json.dumps(response).encode("utf-8") + b"\n"
        )
        self.wfile.flush()
        if evidence.get("replay_mutation_applied"):
            coordinator.mutation_applied = True
        coordinator.request_log.append({
            **evidence, "original_request": original, "request": prepared,
            "classification": classification,
            "pytest_item_id": item_id,
            "item_request_ordinal": item_ordinal,
            "scenario_id": classification["scenario_id"] if classification else None,
        })

        return True


class OfflineRouterServer(
    socketserver.ThreadingUnixStreamServer
):
    daemon_threads = True
    allow_reuse_address = True


def parse_pytest_item_outcomes(stdout):
    prefix = "STAGE06_ITEM_OUTCOMES="
    records = [line[len(prefix):] for line in stdout.splitlines() if line.startswith(prefix)]
    if len(records) != 1:
        return None
    try:
        outcomes = json.loads(records[0])
    except ValueError:
        return None
    allowed = {"passed", "failed", "error", "skipped", "xfailed", "xpassed"}
    if not isinstance(outcomes, dict) or any(
        not isinstance(value, str) or value not in allowed for value in outcomes.values()
    ):
        return None
    return outcomes


def parse_pytest_assertion_failures(stdout, outcomes):
    prefix = "STAGE06_ASSERTION_FAILURES="
    records = [line[len(prefix):] for line in stdout.splitlines() if line.startswith(prefix)]
    if outcomes is None or len(records) != 1:
        return None
    try:
        items = json.loads(records[0])
    except ValueError:
        return None
    if not isinstance(items, list) or any(
        not isinstance(item, str) or outcomes.get(item) != "failed" for item in items
    ) or len(set(items)) != len(items):
        return None
    return items


def evaluate_isolated(
    source,
    route,
    phase="run",
    timeout=30,
    maximum_output=200000,
):
    qualify_os_boundary()

    if phase not in (
        "collect",
        "run",
    ):
        raise ValueError(
            "Invalid evaluation phase"
        )

    if (
        not isinstance(source, str)
        or len(source.encode("utf-8")) > 2_000_000
    ):
        raise ValueError(
            "Generated source too large or invalid"
        )

    started = time.monotonic()
    coordinator = make_router_coordinator(route, phase)

    with tempfile.TemporaryDirectory(
        prefix="stage06_worker_"
    ) as temporary_directory:
        workspace = Path(temporary_directory)
        worker_root, work_dir = _create_worker_root(workspace)

        (work_dir / "generated_test.py").write_text(
            source,
            encoding="utf-8",
        )

        (work_dir / "bootstrap.py").write_text(
            SANDBOX_BOOTSTRAP,
            encoding="utf-8",
        )

        # Explicitly readable after dropping privileges, even with umask 077.
        (work_dir / "generated_test.py").chmod(0o644)
        (work_dir / "bootstrap.py").chmod(0o644)
        host_socket, child_socket = socket.socketpair(
            socket.AF_UNIX, socket.SOCK_STREAM
        )
        try:
            server_thread = threading.Thread(
                target=OfflineRouterHandler,
                args=(host_socket, None, coordinator),
                daemon=True,
            )
            server_thread.start()

            command = construct_sandbox_command(worker_root, sys.executable)
            command.extend([
                sys.executable, "-I", "-B", "-S", "/work/bootstrap.py",
            ])
            try:
                result = bounded_process(
                    command, timeout, maximum_output,
                    phase=phase,
                    router_fd=child_socket.fileno(),
                )
            finally:
                child_socket.close()
                server_thread.join(timeout=3)
        finally:
            host_socket.close()
            if child_socket.fileno() != -1:
                child_socket.close()

        owners = {row.get("failure_owner") for row in coordinator.request_log}
        item_outcomes = parse_pytest_item_outcomes(result["stdout"])
        assertion_failures = parse_pytest_assertion_failures(result["stdout"], item_outcomes)
        if ("EVALUATOR_FAILURE" in owners or result["exit_code"] in {3, 4}
                or "STAGE06_PYTEST_READY" not in result["stdout"].splitlines()
                or "SECURITY_BLOCK:" in result["stderr"]):
            status = "EVALUATOR_FAILURE"
        elif "EVALUATOR_LIMITATION" in owners:
            status = "EVALUATOR_UNAVAILABLE"
        elif result["timed_out"]:
            status = "TIMEOUT"
        elif result["output_truncated"]:
            status = "OUTPUT_LIMIT"
        elif "GENERATED_OUTPUT" in owners:
            status = "TEST_FAILURE"
        elif phase == "run" and assertion_failures is None:
            status = "EVALUATOR_FAILURE"
        elif result["exit_code"] == 0:
            if phase == "collect":
                status = "PASS"
            elif item_outcomes is None:
                status = "EVALUATOR_FAILURE"
            else:
                outcomes = set(item_outcomes.values())
                status = ("PASS" if "passed" in outcomes
                          and outcomes <= {"passed", "skipped", "xfailed"}
                          else "TEST_FAILURE")
        elif phase == "run" and item_outcomes is None:
            status = "EVALUATOR_FAILURE"
        else:
            status = "TEST_FAILURE"

        return {
            "phase": phase,
            "item_outcomes": item_outcomes,
            "assertion_failures": assertion_failures,
            "status": status,
            "exit_code": result["exit_code"],
            "stdout": result["stdout"],
            "stderr": result["stderr"],
            "output_truncated": (
                result["output_truncated"]
            ),
            "duration_seconds": round(
                time.monotonic() - started,
                3,
            ),
            "request_attempts": (
                coordinator.request_log
            ),
        }

# END_STAGE06_WORKER_MODULE

from pathlib import Path

import importlib
import json
import time


CELL11_STARTED = time.time()

require_cell_pass(10)

worker_module_path = (
    PROJECT_ROOT
    / "src/worker.py"
)

worker_write_status = freeze_visible_module(
    "WORKER",
    worker_module_path,
)

importlib.invalidate_caches()

import worker

if worker.WORKER_VERSION != WORKER_VERSION:
    raise RuntimeError(
        "Worker version mismatch."
    )

cell11_report = save_cell_report(
    11,
    worker_version=WORKER_VERSION,
    worker_module_sha256=sha256_file(
        worker_module_path
    ),
    module_integrity="HASH_VERIFIED",
    sandbox_qualification_pending=True,
    hidden_oracles_mounted=False,
)

print(
    json.dumps(
        cell11_report,
        indent=2,
    )
)

print("\nNOTEBOOK 06 — CELL 11: PASS")


{
  "worker_version": "stage06-isolated-worker-v11-single-opportunity",
  "worker_module_sha256": "18236e9aeff6867f2359f1f49764be4e9e1bf7dad8caa78bb6dd627e0f062d22",
  "module_integrity": "HASH_VERIFIED",
  "sandbox_qualification_pending": true,
  "hidden_oracles_mounted": false,
  "status": "PASS",
  "notebook": "06",
  "cell": 11
}

NOTEBOOK 06 — CELL 11: PASS


In [12]:
# ============================================================
# NOTEBOOK 06 — CELL 12
# Verify OS-backed isolation in Kaggle without Linux user namespaces.
# No host execution fallback, silent security downgrade, or invented PASS.
# ============================================================

from __future__ import annotations

import json
import time

import worker


CELL12_STARTED = time.time()

require_cell_pass(11)

# This performs a real chroot, drops to UID/GID 65534 and uses a minimal
# sealed Python runtime. Cell12 then tests the libseccomp filters against
# outbound sockets, subprocesses, hidden-oracle files and resource abuse.
# A Kaggle container without CAP_SYS_CHROOT remains a true SECURITY_BLOCK.
isolation_preflight = worker.qualify_os_boundary()

fixture_route = {
    "scenarios": [
        {
            "scenario_id": "s1",
            "applicability": "APPLICABLE",
            "method": "get",
            "path_template": "/v1/ok",
            "parameter_values": {},
            "expected_validity": "VALID",
            "intent": "valid_baseline_request",
            "documented_response_mapping": {"status": 200},
        }
    ],
    "operation_document": {
        "openapi": "3.0.3",
        "paths": {
            "/v1/ok": {
                "get": {
                    "responses": {
                        "200": {
                            "description": "Successful offline isolation fixture response",
                            "content": {
                                "application/json": {
                                    "schema": {
                                        "type": "object",
                                        "required": ["ok"],
                                        "properties": {"ok": {"type": "boolean", "enum": [True]}},
                                    },
                                },
                            },
                        },
                    },
                },
            },
        },
    },
    "baselines": [
        {
            "scenario_id": "s1",
            "baseline_id": "b1",
            "status": 200,
            "body": {"ok": True},
            "headers": {"Content-Type": "application/json"},
        }
    ],
}


def cell12_check_isolation(name, code, expected_success, timeout=12, maximum_output=200000):
    result = worker.evaluate_isolated(
        code,
        fixture_route,
        timeout=timeout,
        maximum_output=maximum_output,
    )
    expected_status = ("TIMEOUT" if name == "runaway_timeout"
                       else "OUTPUT_LIMIT" if name == "runaway_output"
                       else "PASS" if expected_success else "TEST_FAILURE")
    status = "PASS" if result["status"] == expected_status else "FAIL"
    isolation_fixtures.append({
        "fixture": name,
        "status": status,
        "worker_status": result["status"],
    })
    if status != "PASS":
        diagnostic = json.dumps({
            "stdout_tail": result.get("stdout", "")[-6000:],
            "stderr_tail": result.get("stderr", "")[-6000:],
            "request_attempts": result.get("request_attempts", []),
            "exit_code": result.get("exit_code"),
        }, indent=2, ensure_ascii=False)
        raise RuntimeError(
            f"Isolation fixture {name} failed; worker={result['status']}\n"
            + diagnostic
        )
    return result


isolation_fixtures = []

cell12_check_isolation(
    "allowed_offline_request",
    "import requests\n"
    "def test_request():\n"
    "    r = requests.get('http://openapi.test/v1/ok')\n"
    "    assert r.status_code == 200\n"
    "    assert r.json() == {'ok': True}\n",
    expected_success=True,
)

cell12_check_isolation(
    "outbound_network_denied",
    "import socket\n"
    "def test_network_escape():\n"
    "    socket.create_connection(('1.1.1.1', 80), timeout=2)\n",
    expected_success=False,
)

cell12_check_isolation(
    "hidden_oracle_filesystem_denied",
    "from pathlib import Path\n"
    "def test_hidden_oracle_access():\n"
    "    assert Path(" + repr(str(PROJECT_ROOT / "oracles/frozen")) + ").exists()\n",
    expected_success=False,
)

cell12_check_isolation(
    "subprocess_escape_denied",
    "import subprocess\n"
    "import sys\n"
    "def test_subprocess_escape():\n"
    "    subprocess.run([sys.executable, '-c', 'print(1)'], check=True)\n",
    expected_success=False,
)

cell12_check_isolation(
    "environment_secret_denied",
    "import os\n"
    "def test_environment_secret():\n"
    "    assert 'KAGGLE_KEY' in os.environ\n",
    expected_success=False,
)

timeout_result = cell12_check_isolation(
    "runaway_timeout",
    "def test_runaway_loop():\n    while True:\n        pass\n",
    expected_success=False,
    timeout=3,
)
if timeout_result["status"] != "TIMEOUT":
    raise RuntimeError("Isolation timeout gate did not trigger")

output_result = cell12_check_isolation(
    "runaway_output",
    "def test_large_output():\n    print('z' * 1000000)\n",
    expected_success=False,
    maximum_output=10000,
)
if not output_result["output_truncated"]:
    raise RuntimeError("Isolation output-limit gate did not trigger")


isolation_report = {
    "status": "PASS",
    "worker_version": worker.WORKER_VERSION,
    "preflight": isolation_preflight,
    "fixture_count": len(isolation_fixtures),
    "fixtures": isolation_fixtures,
}

write_json_immutable(
    PROJECT_ROOT / "reports/qualification/isolation_preflight.json",
    isolation_preflight,
)
write_json_immutable(
    PROJECT_ROOT / "reports/qualification/isolation_fixtures.json",
    isolation_report,
)

cell12_report = save_cell_report(
    12,
    isolation_fixture_count=len(isolation_fixtures),
    os_boundary_qualified=True,
    hidden_oracle_access="DENIED",
    outbound_network_access="DENIED",
    unauthorized_process_execution="DENIED",
)

print(json.dumps(cell12_report, indent=2))
print("\nNOTEBOOK 06 — CELL 12: PASS")


{
  "isolation_fixture_count": 7,
  "os_boundary_qualified": true,
  "hidden_oracle_access": "DENIED",
  "outbound_network_access": "DENIED",
  "unauthorized_process_execution": "DENIED",
  "status": "PASS",
  "notebook": "06",
  "cell": 12
}

NOTEBOOK 06 — CELL 12: PASS


In [13]:
# NOTEBOOK 06 — CELL 13
# Qualify the full synthetic evaluation path before real outputs.

from __future__ import annotations

import json
import time

import worker


CELL13_STARTED = time.time()

require_cell_pass(12)

synthetic_scenario = {
    "scenario_id": "fixture_valid",
    "applicability": "APPLICABLE",
    "method": "post",
    "path_template": "/users",
    "parameter_values": {},
    "body": {
        "age": 25,
    },
    "expected_validity": "VALID",
    "intent": "valid_baseline_request",
    "documented_response_mapping": {"status": 201},
}

synthetic_baseline = {
    "baseline_id": "fixture_baseline",
    "scenario_id": "fixture_valid",
    "status": 201,
    "body": {
        "id": "u1",
    },
    "headers": {
        "Content-Type": "application/json",
    },
}

synthetic_route = {
    "scenarios": [
        synthetic_scenario
    ],
    "baselines": [
        synthetic_baseline
    ],
    "operation_document": {
        "openapi": "3.0.3",
        "paths": {"/users": {"post": {
            "responses": {"201": {"description": "Fixture success"}}, "requestBody": {
            "required": True,
            "content": {"application/json": {"schema": {
                "type": "object", "required": ["age"],
                "properties": {"age": {"type": "integer", "minimum": 18, "maximum": 60}},
            }}},
        }}}},
    },
}

correct_test = """
import requests

def test_create_user():
    response = requests.post(
        "http://openapi.test/users",
        json={"age": 25},
    )
    assert response.status_code == 201
    assert isinstance(
        response.json()["id"],
        str,
    )
"""

rigid_test = """
import requests

def test_create_user():
    response = requests.post(
        "http://openapi.test/users",
        json={"age": 25},
    )
    assert response.status_code == 201
    assert response.json()["id"] == "u1"
"""

status_only_test = """
import requests

def test_create_user():
    response = requests.post(
        "http://openapi.test/users",
        json={"age": 25},
    )
    assert response.status_code == 201
"""

parametrized_test = """
import pytest

@pytest.mark.parametrize("value", [1, 2])
def test_values(value):
    assert value in (1, 2)
"""

alternative_valid_test = correct_test.replace('{"age": 25}', '{"age": 41}')
invalid_unmapped_test = correct_test.replace('{"age": 25}', '{"age": "old"}')

synthetic_variation = {
    "variation_id": "fixture_variation",
    "baseline_id": "fixture_baseline",
    "scenario_id": "fixture_valid",
    "status": 201,
    "body": {
        "id": "u2",
    },
}

fault_wrong_type = {
    "fault_id": "fixture_wrong_type",
    "baseline_id": "fixture_baseline",
    "scenario_id": "fixture_valid",
    "status": 201,
    "body": {
        "id": 7,
    },
}

fault_missing_required = {
    "fault_id": "fixture_missing_required",
    "baseline_id": "fixture_baseline",
    "scenario_id": "fixture_valid",
    "status": 201,
    "body": {},
}

reference_run = worker.evaluate_isolated(correct_test, synthetic_route, timeout=25)
if (reference_run["status"] != "PASS" or len(reference_run["request_attempts"]) != 1
        or reference_run["request_attempts"][0]["status"] != "DELIVERED"):
    raise RuntimeError("Could not capture the reference request for replay qualification")
reference_request = reference_run["request_attempts"][0]
replay_target = {key: reference_request[key] for key in (
    "baseline_id", "scenario_id", "pytest_item_id", "item_request_ordinal", "request",
)}

repeat_test = """
import requests

def test_create_user():
    url = "http://openapi.test/users"
    requests.post(url, json={"age": 25})
    assert requests.post(url, json={"age": 25}).json()["id"] == "u1"
    assert requests.post(url, json={"age": 25}).json()["id"] == "u1"
"""

integrated_cases = [
    {
        "name": "baseline_pass",
        "source": correct_test,
        "route": synthetic_route,
        "expected": "PASS",
    },
    {
        "name": "baseline_assertion_failure",
        "source": correct_test.replace(
            "== 201",
            "== 400",
        ),
        "route": synthetic_route,
        "expected": "TEST_FAILURE",
    },
    {
        "name": "alternative_valid_body_not_fixture_equality",
        "source": alternative_valid_test,
        "route": synthetic_route,
        "expected": "PASS",
    },
    {
        "name": "unmapped_invalid_is_evaluator_unavailable",
        "source": invalid_unmapped_test,
        "route": synthetic_route,
        "expected": "EVALUATOR_UNAVAILABLE",
    },
    {
        "name": "parametrization",
        "source": parametrized_test,
        "route": synthetic_route,
        "expected": "PASS",
    },
    {
        "name": "syntax_error",
        "source": "def test_x(: pass",
        "route": synthetic_route,
        "expected": "TEST_FAILURE",
    },
    {
        "name": "import_error",
        "source": (
            "import nonexistent_stage06_module\n"
            "def test_x(): assert True"
        ),
        "route": synthetic_route,
        "expected": "TEST_FAILURE",
    },
    {
        "name": "valid_variation_accepted",
        "source": correct_test,
        "route": {
            **synthetic_route,
            "mode": "variation",
            "mutation": synthetic_variation,
        },
        "expected": "PASS",
    },
    {
        "name": "valid_variation_rejected_by_rigid_test",
        "source": rigid_test,
        "route": {
            **synthetic_route,
            "mode": "variation",
            "mutation": synthetic_variation,
        },
        "expected": "TEST_FAILURE",
    },
    {
        "name": "wrong_type_fault_killed",
        "source": correct_test,
        "route": {
            **synthetic_route,
            "mode": "fault",
            "mutation": fault_wrong_type,
        },
        "expected": "TEST_FAILURE",
    },
    {
        "name": "missing_required_fault_killed",
        "source": correct_test,
        "route": {
            **synthetic_route,
            "mode": "fault",
            "mutation": fault_missing_required,
        },
        "expected": "TEST_FAILURE",
    },
    {
        "name": "status_only_fault_survives",
        "source": status_only_test,
        "route": {
            **synthetic_route,
            "mode": "fault",
            "mutation": fault_wrong_type,
        },
        "expected": "PASS",
    },
    {
        "name": "fault_not_consumed",
        "source": (
            "def test_x():\n"
            "    assert True\n"
        ),
        "route": {
            **synthetic_route,
            "mode": "fault",
            "mutation": fault_wrong_type,
        },
        "expected": "PASS",
    },
    {
        "name": "same_request_three_times_mutates_only_first",
        "source": repeat_test,
        "route": {**synthetic_route, "mode": "fault", "mutation": fault_wrong_type},
        "expected": "PASS",
    },
    {
        "name": "replay_can_target_the_second_occurrence",
        "source": repeat_test,
        "route": {**synthetic_route, "mode": "fault", "mutation": fault_wrong_type,
                  "replay_target": {**replay_target, "item_request_ordinal": 2}},
        "expected": "TEST_FAILURE",
    },
    {
        "name": "later_identical_request_cannot_take_target",
        "source": repeat_test.replace('requests.post(url, json={"age": 25})',
                                      'requests.post(url, json={"age": 41})', 1),
        "route": {**synthetic_route, "mode": "fault", "mutation": fault_wrong_type},
        "expected": "PASS",
    },
]

integrated_cases.extend([
    {"name": "all_skipped_is_baseline_failure", "route": synthetic_route,
     "source": "import pytest\n@pytest.mark.skip\ndef test_x(): pass\n",
     "expected": "TEST_FAILURE"},
    {"name": "non_strict_xpass_is_baseline_failure", "route": synthetic_route,
     "source": "import pytest\n@pytest.mark.xfail\ndef test_x(): assert True\n",
     "expected": "TEST_FAILURE"},
])
integrated_cases.append({
    "name": "separate_item_request_and_assertion_attribution",
    "source": (
        "import requests\n"
        "def test_a_request():\n"
        "    assert requests.post('http://openapi.test/users', json={'age': 25}).status_code == 201\n"
        "def test_b_other_assertion():\n"
        "    assert False\n"
    ),
    "route": synthetic_route,
    "expected": "TEST_FAILURE",
})
integrated_results = []

for index, case in enumerate(
    integrated_cases,
    start=1,
):
    route = case["route"]
    if route.get("mode") in {"fault", "variation"} and "replay_target" not in route:
        route = {**route, "replay_target": replay_target}
    result = worker.evaluate_isolated(
        case["source"],
        route,
        timeout=25,
    )

    consumed = any(
        attempt.get("status") == "DELIVERED"
        for attempt in result["request_attempts"]
    )

    observed = result["status"]
    if case["name"] == "separate_item_request_and_assertion_attribution":
        attempts = result["request_attempts"]
        failures = result.get("assertion_failures") or []
        if (len(attempts) != 1 or not attempts[0].get("pytest_item_id", "").endswith("::test_a_request")
                or len(failures) != 1 or not failures[0].endswith("::test_b_other_assertion")):
            observed = "UNQUALIFIED_ITEM_ATTRIBUTION"
    if case["name"] == "unmapped_invalid_is_evaluator_unavailable":
        if not any(row.get("original_request") and row.get("request")
                   and row.get("failure_owner") == "EVALUATOR_LIMITATION"
                   for row in result["request_attempts"]):
            observed = "MISSING_UNAVAILABLE_EVIDENCE"

    if (
        case["name"] == "fault_not_consumed"
        and consumed
    ):
        observed = "INVALID_FAULT_CONSUMPTION"

    if "replay_target" in route:
        attempts = result["request_attempts"]
        injected_ordinals = [row["item_request_ordinal"] for row in attempts
                             if row.get("replay_mutation_applied")]
        no_injection = case["name"] in {
            "fault_not_consumed", "later_identical_request_cannot_take_target",
        }
        expected_ordinals = [] if no_injection else [route["replay_target"]["item_request_ordinal"]]
        if injected_ordinals != expected_ordinals:
            observed = "INVALID_REPLAY_OCCURRENCE"
        if case["name"] == "same_request_three_times_mutates_only_first" and len(attempts) != 3:
            observed = "MISSING_REPEATED_REQUESTS"

    passed = (
        observed == case["expected"]
    )

    integrated_results.append({
        "name": case["name"],
        "expected": case["expected"],
        "observed": observed,
        "consumed_response": consumed,
        "status": (
            "PASS"
            if passed
            else "FAIL"
        ),
    })

    print({
        "fixture": case["name"],
        "processed": index,
        "total": len(integrated_cases),
        "status": (
            "PASS"
            if passed
            else "FAIL"
        ),
    })

    if not passed:
        raise RuntimeError(
            "Integrated evaluator fixture failed: "
            + case["name"]
            + "\n"
            + result["stderr"][:1200]
        )

integrated_report = {
    "status": "PASS",
    "fixture_count": len(
        integrated_results
    ),
    "fixtures": integrated_results,
    "worker_sha256": sha256_file(
        PROJECT_ROOT
        / "src/worker.py"
    ),
    "routing_sha256": sha256_file(
        PROJECT_ROOT
        / "src/routing.py"
    ),
}

write_json_immutable(
    PROJECT_ROOT
    / "reports/qualification/"
    "integrated_evaluator_fixtures.json",
    integrated_report,
)

cell13_report = save_cell_report(
    13,
    integrated_fixture_count=len(
        integrated_results
    ),
    qualification_status="PASS",
    generated_model_outputs_used=False,
)

print(
    json.dumps(
        cell13_report,
        indent=2,
    )
)

print("\nNOTEBOOK 06 — CELL 13: PASS")


{'fixture': 'baseline_pass', 'processed': 1, 'total': 19, 'status': 'PASS'}
{'fixture': 'baseline_assertion_failure', 'processed': 2, 'total': 19, 'status': 'PASS'}
{'fixture': 'alternative_valid_body_not_fixture_equality', 'processed': 3, 'total': 19, 'status': 'PASS'}
{'fixture': 'unmapped_invalid_is_evaluator_unavailable', 'processed': 4, 'total': 19, 'status': 'PASS'}
{'fixture': 'parametrization', 'processed': 5, 'total': 19, 'status': 'PASS'}
{'fixture': 'syntax_error', 'processed': 6, 'total': 19, 'status': 'PASS'}
{'fixture': 'import_error', 'processed': 7, 'total': 19, 'status': 'PASS'}
{'fixture': 'valid_variation_accepted', 'processed': 8, 'total': 19, 'status': 'PASS'}
{'fixture': 'valid_variation_rejected_by_rigid_test', 'processed': 9, 'total': 19, 'status': 'PASS'}
{'fixture': 'wrong_type_fault_killed', 'processed': 10, 'total': 19, 'status': 'PASS'}
{'fixture': 'missing_required_fault_killed', 'processed': 11, 'total': 19, 'status': 'PASS'}
{'fixture': 'status_only_faul

In [14]:
# NOTEBOOK 06 — CELL 14
# Render the exact frozen Stage 00 prompt for both conditions.

from __future__ import annotations

from pathlib import Path

import hashlib
import json
import time
import yaml

from transformers import AutoTokenizer


CELL14_STARTED = time.time()

require_cell_pass(13)

runtime = yaml.safe_load(
    (
        PROJECT_ROOT
        / "config/runtime.yaml"
    ).read_text(
        encoding="utf-8"
    )
)

study = yaml.safe_load(
    (
        PROJECT_ROOT
        / "config/study.yaml"
    ).read_text(
        encoding="utf-8"
    )
)

EXPECTED_TOKENIZER_REVISION = (
    "c03e6d358207e414f1eca0bb1891e29f1db0e242"
)

tokenizer_revision = runtime[
    "resolved_tokenizer_revision"
]

if tokenizer_revision != EXPECTED_TOKENIZER_REVISION:
    raise RuntimeError(
        "Frozen tokenizer revision mismatch."
    )

tokenizer = AutoTokenizer.from_pretrained(
    runtime["model_id"],
    revision=tokenizer_revision,
    trust_remote_code=False,
)

if not tokenizer.chat_template:
    raise RuntimeError(
        "Frozen tokenizer chat template unavailable."
    )

system_template = (
    PROJECT_ROOT
    / "config/system_prompt.txt"
).read_text(
    encoding="utf-8"
)

user_template = (
    PROJECT_ROOT
    / "config/user_prompt.txt"
).read_text(
    encoding="utf-8"
)

selection = read_json(
    PROJECT_ROOT
    / "data/inventory/final_selected_operations.json"
)

reconstruction_by_id = {
    row["op_id"]: row
    for row in read_jsonl_file(
        PROJECT_ROOT
        / "data/audits/"
        "cell04_pair_reconstruction.jsonl"
    )
}

if len(reconstruction_by_id) != 100:
    raise RuntimeError(
        "Incomplete reconstructed-input hash ledger."
    )

rendered_prompt_ledger = []

for index, operation in enumerate(
    selection,
    start=1,
):
    op_id = operation["op_id"]

    boundary = hashlib.sha256(
        (
            f"{study['study_id']}|{op_id}"
        ).encode("utf-8")
    ).hexdigest()[:24]

    for condition in (
        "L0",
        "L1",
    ):
        source_path = (
            PROJECT_ROOT
            / "data/inputs"
            / condition
            / f"{op_id}.json"
        )

        source = read_json(
            source_path
        )

        canonical_json = json.dumps(
            source,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        )

        if boundary in canonical_json:
            raise RuntimeError(
                "Prompt boundary collision: "
                + op_id
            )

        user_message = (
            user_template.replace(
                "{BOUNDARY}",
                boundary,
            ).replace(
                "{CANONICAL_OPENAPI_JSON}",
                canonical_json,
            )
        )

        if (
            "{BOUNDARY}" in user_message
            or "{CANONICAL_OPENAPI_JSON}"
            in user_message
        ):
            raise RuntimeError(
                "Unrendered prompt template token."
            )

        rendered = (
            tokenizer.apply_chat_template(
                [
                    {
                        "role": "system",
                        "content": system_template,
                    },
                    {
                        "role": "user",
                        "content": user_message,
                    },
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
        )

        rendered_bytes = rendered.encode(
            "utf-8"
        )

        relative_path = (
            f"data/prompts/pilot/"
            f"{op_id}/{condition}.txt"
        )

        destination = (
            PROJECT_ROOT
            / relative_path
        )

        write_immutable(
            destination,
            rendered_bytes,
        )

        rendered_prompt_ledger.append({
            "op_id": op_id,
            "pair_id": operation["pair_id"],
            "stratum": operation["stratum"],
            "condition": condition,
            "boundary": boundary,
            "relative_path": relative_path,
            "prompt_sha256": sha256_bytes(
                rendered_bytes
            ),
            "input_sha256": reconstruction_by_id[
                op_id
            ][
                f"{condition}_sha256"
            ],
        })

    if index % 20 == 0:
        print({
            "pairs_rendered": index,
            "total_pairs": 100,
        })

if len(rendered_prompt_ledger) != 200:
    raise RuntimeError(
        "Expected exactly 200 frozen prompts."
    )

ledger_path = (
    PROJECT_ROOT
    / "data/audits/"
    "cell14_rendered_prompt_ledger.jsonl"
)

write_immutable(
    ledger_path,
    canonical_jsonl_bytes(
        rendered_prompt_ledger
    ),
)

cell14_report = save_cell_report(
    14,
    rendered_prompt_count=200,
    tokenizer_revision=tokenizer_revision,
    rendered_prompt_ledger_sha256=sha256_file(
        ledger_path
    ),
)

print(
    json.dumps(
        cell14_report,
        indent=2,
    )
)

print("\nNOTEBOOK 06 — CELL 14: PASS")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

{'pairs_rendered': 20, 'total_pairs': 100}
{'pairs_rendered': 40, 'total_pairs': 100}
{'pairs_rendered': 60, 'total_pairs': 100}
{'pairs_rendered': 80, 'total_pairs': 100}
{'pairs_rendered': 100, 'total_pairs': 100}
{
  "rendered_prompt_count": 200,
  "tokenizer_revision": "c03e6d358207e414f1eca0bb1891e29f1db0e242",
  "rendered_prompt_ledger_sha256": "e12202c8d73b496dd6299dbb154e5ce7e63ef829b8e7f326ec5084fa6a7328c8",
  "status": "PASS",
  "notebook": "06",
  "cell": 14
}

NOTEBOOK 06 — CELL 14: PASS


In [15]:
# NOTEBOOK 06 — CELL 15
# Tokenize and verify all rendered prompts.

from __future__ import annotations

import hashlib
import json
import time


CELL15_STARTED = time.time()

require_cell_pass(14)

MAX_RENDERED_INPUT_TOKENS = 5120

rendered_rows = read_jsonl_file(
    PROJECT_ROOT
    / "data/audits/"
    "cell14_rendered_prompt_ledger.jsonl"
)

renderer_fingerprint = sha256_bytes(
    (
        tokenizer.chat_template
        + "\n"
        + tokenizer_revision
        + "\n"
        + system_template
        + "\n"
        + user_template
    ).encode("utf-8")
)

token_ledger = []

for index, row in enumerate(
    rendered_rows,
    start=1,
):
    prompt_path = (
        PROJECT_ROOT
        / row["relative_path"]
    )

    prompt_bytes = prompt_path.read_bytes()

    if (
        sha256_bytes(prompt_bytes)
        != row["prompt_sha256"]
    ):
        raise RuntimeError(
            "Rendered prompt hash mismatch: "
            + row["op_id"]
        )

    rendered_text = prompt_bytes.decode(
        "utf-8"
    )

    token_ids = tokenizer.encode(
        rendered_text,
        add_special_tokens=False,
    )

    token_count = len(
        token_ids
    )

    if (
        token_count == 0
        or token_count > MAX_RENDERED_INPUT_TOKENS
    ):
        raise RuntimeError(
            "Frozen prompt token budget exceeded: "
            + row["op_id"]
            + "/"
            + row["condition"]
        )

    token_ids_bytes = json.dumps(
        token_ids,
        separators=(",", ":"),
    ).encode("utf-8")

    token_ledger.append({
        **row,
        "prompt_tokens": token_count,
        "token_ids_sha256": sha256_bytes(
            token_ids_bytes
        ),
        "renderer_fingerprint": (
            renderer_fingerprint
        ),
    })

    if index % 40 == 0:
        print({
            "tokenized_prompts": index,
            "total_prompts": 200,
        })

if len(token_ledger) != 200:
    raise RuntimeError(
        "Incomplete prompt token ledger."
    )

pairs = {}

for row in token_ledger:
    pairs.setdefault(
        row["pair_id"],
        [],
    ).append(row)

if len(pairs) != 100:
    raise RuntimeError(
        "Expected exactly 100 prompt pairs."
    )

for pair_id, rows in pairs.items():
    if (
        len(rows) != 2
        or {
            row["condition"]
            for row in rows
        } != {"L0", "L1"}
    ):
        raise RuntimeError(
            "Invalid prompt pair: "
            + pair_id
        )

    if (
        rows[0]["boundary"]
        != rows[1]["boundary"]
    ):
        raise RuntimeError(
            "Prompt boundary mismatch: "
            + pair_id
        )

    if (
        rows[0]["prompt_sha256"]
        == rows[1]["prompt_sha256"]
    ):
        raise RuntimeError(
            "Unexpected identical condition prompts: "
            + pair_id
        )

ledger_path = (
    PROJECT_ROOT
    / "data/audits/"
    "cell15_prompt_token_ledger.jsonl"
)

write_immutable(
    ledger_path,
    canonical_jsonl_bytes(
        token_ledger
    ),
)

cell15_report = save_cell_report(
    15,
    tokenized_prompt_count=200,
    verified_pair_count=100,
    maximum_rendered_input_tokens=max(
        row["prompt_tokens"]
        for row in token_ledger
    ),
    tokenizer_revision=tokenizer_revision,
    renderer_fingerprint=renderer_fingerprint,
    token_ledger_sha256=sha256_file(
        ledger_path
    ),
)

print(
    json.dumps(
        cell15_report,
        indent=2,
    )
)

print("\nNOTEBOOK 06 — CELL 15: PASS")

{'tokenized_prompts': 40, 'total_prompts': 200}
{'tokenized_prompts': 80, 'total_prompts': 200}
{'tokenized_prompts': 120, 'total_prompts': 200}
{'tokenized_prompts': 160, 'total_prompts': 200}
{'tokenized_prompts': 200, 'total_prompts': 200}
{
  "tokenized_prompt_count": 200,
  "verified_pair_count": 100,
  "maximum_rendered_input_tokens": 4706,
  "tokenizer_revision": "c03e6d358207e414f1eca0bb1891e29f1db0e242",
  "renderer_fingerprint": "7c389152ef112f86dbd1f2da7341436278420ee5115547bf6152bde8904e43b1",
  "token_ledger_sha256": "372e5875e0660e368bbfdad22b9e05ad2bd50cf9771382c7f2e83888892fb9f5",
  "status": "PASS",
  "notebook": "06",
  "cell": 15
}

NOTEBOOK 06 — CELL 15: PASS


In [16]:
# NOTEBOOK 06 — CELL 16
# Deterministic seeds and balanced condition order.

from __future__ import annotations

from collections import defaultdict, Counter
from pathlib import Path

import csv
import hashlib
import io
import json
import time


CELL16_STARTED = time.time()

require_cell_pass(15)

GENERATION_SEED_BASE = 20261104

MODEL_REVISION = runtime[
    "resolved_model_revision"
]

TASK_FIELDS = [
    "task_id",
    "pair_id",
    "op_id",
    "partition",
    "condition",
    "replicate",
    "seed",
    "order",
    "model_revision",
    "prompt_path",
    "prompt_sha256",
    "prompt_tokens",
    "expected_output_path",
]

prompt_ledger = read_jsonl_file(
    PROJECT_ROOT
    / "data/audits/"
    "cell15_prompt_token_ledger.jsonl"
)

prompts_by_operation = defaultdict(
    dict
)

for row in prompt_ledger:
    prompts_by_operation[
        row["op_id"]
    ][
        row["condition"]
    ] = row

selection = read_json(
    PROJECT_ROOT
    / "data/inventory/final_selected_operations.json"
)

operations_by_stratum = defaultdict(
    list
)

for row in selection:
    operations_by_stratum[
        row["stratum"]
    ].append(row)

tasks = []

for stratum in (
    "low",
    "medium",
    "high",
):
    ordered_operations = sorted(
        operations_by_stratum[stratum],
        key=lambda row: hashlib.sha256(
            (
                "stage06-order|"
                + row["pair_id"]
                + "|20261104"
            ).encode("utf-8")
        ).hexdigest(),
    )

    for index, operation in enumerate(
        ordered_operations
    ):
        pair_id = operation["pair_id"]
        op_id = operation["op_id"]

        seed_material = (
            f"{study['protocol_version']}|"
            f"pilot|{pair_id}|"
            f"{GENERATION_SEED_BASE}"
        )

        pair_seed = int.from_bytes(
            hashlib.sha256(
                seed_material.encode("utf-8")
            ).digest()[:8],
            "big",
        )

        first_condition = (
            "L0"
            if index % 2 == 0
            else "L1"
        )

        second_condition = (
            "L1"
            if first_condition == "L0"
            else "L0"
        )

        for condition in (
            first_condition,
            second_condition,
        ):
            prompt = (
                prompts_by_operation[
                    op_id
                ][condition]
            )

            task_identity = (
                f"pilot|{pair_id}|"
                f"{condition}|1"
            )

            task_id = (
                "task_"
                + hashlib.sha256(
                    task_identity.encode("utf-8")
                ).hexdigest()[:24]
            )

            tasks.append({
                "task_id": task_id,
                "pair_id": pair_id,
                "op_id": op_id,
                "partition": "pilot",
                "condition": condition,
                "replicate": 1,
                "seed": pair_seed,
                "order": len(tasks) + 1,
                "model_revision": MODEL_REVISION,
                "prompt_path": prompt[
                    "relative_path"
                ],
                "prompt_sha256": prompt[
                    "prompt_sha256"
                ],
                "prompt_tokens": prompt[
                    "prompt_tokens"
                ],
                "expected_output_path": (
                    "runs/pilot/raw/"
                    + task_id
                    + ".txt"
                ),
            })

if len(tasks) != 200:
    raise RuntimeError(
        "Expected exactly 200 generation tasks."
    )

if len({
    row["task_id"]
    for row in tasks
}) != 200:
    raise RuntimeError(
        "Duplicate task identities."
    )

if len({
    row["expected_output_path"]
    for row in tasks
}) != 200:
    raise RuntimeError(
        "Duplicate expected output paths."
    )

tasks_by_pair = defaultdict(
    list
)

for row in tasks:
    tasks_by_pair[
        row["pair_id"]
    ].append(row)

if len(tasks_by_pair) != 100:
    raise RuntimeError(
        "Expected exactly 100 task pairs."
    )

for pair_id, rows in tasks_by_pair.items():
    if (
        len(rows) != 2
        or {
            row["condition"]
            for row in rows
        } != {"L0", "L1"}
        or rows[0]["seed"] != rows[1]["seed"]
    ):
        raise RuntimeError(
            "Invalid task pair: "
            + pair_id
        )

for stratum, operations in (
    operations_by_stratum.items()
):
    first_conditions = []

    for operation in operations:
        matching_tasks = [
            row
            for row in tasks
            if row["op_id"]
            == operation["op_id"]
        ]

        first_task = min(
            matching_tasks,
            key=lambda row: row["order"],
        )

        first_conditions.append(
            first_task["condition"]
        )

    counts = Counter(
        first_conditions
    )

    if abs(
        counts["L0"]
        - counts["L1"]
    ) > 1:
        raise RuntimeError(
            "Condition-first order is unbalanced: "
            + stratum
        )

csv_buffer = io.StringIO(
    newline=""
)

writer = csv.DictWriter(
    csv_buffer,
    fieldnames=TASK_FIELDS,
    lineterminator="\n",
)

writer.writeheader()
writer.writerows(tasks)

tasks_path = (
    PROJECT_ROOT
    / "data/tasks.csv"
)

write_immutable(
    tasks_path,
    csv_buffer.getvalue(),
)

cell16_report = save_cell_report(
    16,
    task_count=200,
    pair_count=100,
    replicate_count=1,
    balanced_condition_order=True,
    model_revision=MODEL_REVISION,
    task_ledger_sha256=sha256_file(
        tasks_path
    ),
)

print(
    json.dumps(
        cell16_report,
        indent=2,
    )
)

print("\nNOTEBOOK 06 — CELL 16: PASS")

{
  "task_count": 200,
  "pair_count": 100,
  "replicate_count": 1,
  "balanced_condition_order": true,
  "model_revision": "c03e6d358207e414f1eca0bb1891e29f1db0e242",
  "task_ledger_sha256": "b3170527b07cd39dbaf4cdc185d0f1e9e58c48f675dba33250adc9375467b235",
  "status": "PASS",
  "notebook": "06",
  "cell": 16
}

NOTEBOOK 06 — CELL 16: PASS


In [17]:
# NOTEBOOK 06 — CELL 17
# Register statistical analysis and freeze generation preflight.

# BEGIN_STAGE06_STATISTICS_MODULE

from __future__ import annotations

from collections import defaultdict
import math
import random


STATISTICS_VERSION = (
    "stage06-paired-family-cluster-analysis-v1"
)

STRATA = (
    "low",
    "medium",
    "high",
)


def weighted_stratum_effect(
    rows,
    population_shares,
    conditional=False,
):
    """
    Calculate the design-weighted L1-L0 contrast.

    Required row fields:
        stratum, family_id, d, pi

    d is the paired difference.
    pi is the operation inclusion probability.
    """
    usable_rows = [
        row
        for row in rows
        if row.get("d") is not None
    ]

    represented_strata = sorted({
        row["stratum"]
        for row in usable_rows
    })

    if (
        not usable_rows
        or any(
            stratum not in STRATA
            for stratum in represented_strata
        )
    ):
        raise ValueError(
            "No estimable stratified observations."
        )

    stratum_effects = {}
    stratum_weight_totals = {}

    for stratum in represented_strata:
        subset = [
            row
            for row in usable_rows
            if row["stratum"] == stratum
        ]

        weights = [
            1.0 / float(row["pi"])
            for row in subset
        ]

        if (
            not subset
            or min(weights) <= 0
        ):
            raise ValueError(
                "Invalid inclusion probability."
            )

        total_weight = sum(
            weights
        )

        stratum_weight_totals[
            stratum
        ] = total_weight

        stratum_effects[
            stratum
        ] = (
            sum(
                weight * float(row["d"])
                for weight, row in zip(
                    weights,
                    subset,
                )
            )
            / total_weight
        )

    if (
        not conditional
        and set(represented_strata)
        != set(STRATA)
    ):
        raise ValueError(
            "A target stratum is missing."
        )

    if conditional:
        grand_total = sum(
            stratum_weight_totals.values()
        )

        target_weights = {
            stratum: (
                stratum_weight_totals[stratum]
                / grand_total
            )
            for stratum in represented_strata
        }

    else:
        target_weights = {
            stratum: float(
                population_shares[stratum]
            )
            for stratum in represented_strata
        }

    if abs(
        sum(target_weights.values()) - 1
    ) > 1e-9:
        raise ValueError(
            "Target stratum shares do not sum to one."
        )

    effect = sum(
        target_weights[stratum]
        * stratum_effects[stratum]
        for stratum in represented_strata
    )

    return {
        "effect": effect,
        "stratum_effects": stratum_effects,
        "target_shares": target_weights,
        "stratum_weight_totals": (
            stratum_weight_totals
        ),
    }


def family_cluster_standard_error(
    rows,
    population_shares,
    conditional=False,
):
    """
    Family-clustered CR1 sandwich variance.

    Correction:
        G/(G-1) * (n-1)/(n-k)
    """
    usable_rows = [
        row
        for row in rows
        if row.get("d") is not None
    ]

    effect_record = weighted_stratum_effect(
        usable_rows,
        population_shares,
        conditional=conditional,
    )

    stratum_effects = effect_record[
        "stratum_effects"
    ]

    target_weights = effect_record[
        "target_shares"
    ]

    H = effect_record[
        "stratum_weight_totals"
    ]

    n = len(
        usable_rows
    )

    families = {
        row["family_id"]
        for row in usable_rows
    }

    G = len(families)
    k = len(stratum_effects)

    if G <= 1 or n <= k:
        raise ValueError(
            "Insufficient families or residual degrees of freedom."
        )

    family_scores = defaultdict(
        lambda: defaultdict(float)
    )

    for row in usable_rows:
        stratum = row["stratum"]
        family_id = row["family_id"]

        weight = (
            1.0 / float(row["pi"])
        )

        residual = (
            float(row["d"])
            - stratum_effects[stratum]
        )

        family_scores[
            family_id
        ][stratum] += (
            weight * residual
        )

    correction = (
        G / (G - 1)
    ) * (
        (n - 1) / (n - k)
    )

    variance = correction * sum(
        (
            sum(
                target_weights[stratum]
                * scores.get(stratum, 0.0)
                / H[stratum]
                for stratum in stratum_effects
            )
        ) ** 2
        for scores in family_scores.values()
    )

    if (
        variance <= 0
        or not math.isfinite(variance)
    ):
        raise ValueError(
            "Cluster variance is not estimable."
        )

    return {
        **effect_record,
        "standard_error": math.sqrt(
            variance
        ),
        "variance": variance,
        "family_count": G,
        "pair_count": n,
        "small_sample_correction": correction,
        "degrees_of_freedom": G - 1,
    }


def primary_t_inference(
    rows,
    population_shares,
    conditional=False,
):
    from scipy.stats import t

    result = family_cluster_standard_error(
        rows,
        population_shares,
        conditional=conditional,
    )

    df = result[
        "degrees_of_freedom"
    ]

    effect = result[
        "effect"
    ]

    se = result[
        "standard_error"
    ]

    critical = float(
        t.ppf(
            0.975,
            df,
        )
    )

    p_value = float(
        2 * t.sf(
            abs(effect / se),
            df,
        )
    )

    result.update({
        "pointwise_ci_lower": (
            effect - critical * se
        ),
        "pointwise_ci_upper": (
            effect + critical * se
        ),
        "two_sided_p_value": p_value,
    })

    return result


def holm_adjustment(
    p_values,
):
    ordered = sorted(
        p_values,
        key=p_values.get,
    )

    adjusted = {}
    running_maximum = 0.0

    m = len(
        ordered
    )

    for index, endpoint in enumerate(
        ordered
    ):
        p = float(
            p_values[endpoint]
        )

        if not 0 <= p <= 1:
            raise ValueError(
                "Invalid p-value."
            )

        candidate = min(
            1.0,
            (m - index) * p,
        )

        running_maximum = max(
            running_maximum,
            candidate,
        )

        adjusted[endpoint] = (
            running_maximum
        )

    return adjusted


def family_multiplier_interval(
    rows,
    population_shares,
    seed=20261105,
    draws=10000,
    conditional=False,
):
    """
    Family-shared exponential multiplier bootstrap.

    Exponential multipliers have mean one,
    variance one, and remain nonnegative.
    """
    usable_rows = [
        row
        for row in rows
        if row.get("d") is not None
    ]

    families = sorted({
        row["family_id"]
        for row in usable_rows
    })

    if (
        len(families) < 2
        or draws < 100
    ):
        raise ValueError(
            "Insufficient bootstrap information."
        )

    generator = random.Random(
        seed
    )

    bootstrap_effects = []

    for _ in range(draws):
        multipliers = {
            family_id: generator.expovariate(
                1.0
            )
            for family_id in families
        }

        by_stratum = defaultdict(
            lambda: [0.0, 0.0]
        )

        for row in usable_rows:
            weight = (
                multipliers[row["family_id"]]
                / float(row["pi"])
            )

            values = by_stratum[
                row["stratum"]
            ]

            values[0] += (
                weight * float(row["d"])
            )

            values[1] += weight

        if any(
            values[1] <= 0
            for values in by_stratum.values()
        ):
            raise ValueError(
                "Zero bootstrap stratum weight."
            )

        if conditional:
            total_weight = sum(
                values[1]
                for values in by_stratum.values()
            )

            target_weights = {
                stratum: values[1] / total_weight
                for stratum, values
                in by_stratum.items()
            }

        else:
            if (
                set(by_stratum)
                != set(population_shares)
            ):
                raise ValueError(
                    "Bootstrap missing target stratum."
                )

            target_weights = (
                population_shares
            )

        effect = sum(
            float(target_weights[stratum])
            * values[0]
            / values[1]
            for stratum, values
            in by_stratum.items()
        )

        bootstrap_effects.append(
            effect
        )

    bootstrap_effects.sort()

    def quantile(probability):
        index = (
            probability
            * (len(bootstrap_effects) - 1)
        )

        lower = int(
            index
        )

        upper = min(
            lower + 1,
            len(bootstrap_effects) - 1,
        )

        fraction = (
            index - lower
        )

        return (
            bootstrap_effects[lower]
            + fraction
            * (
                bootstrap_effects[upper]
                - bootstrap_effects[lower]
            )
        )

    return {
        "lower": quantile(0.025),
        "upper": quantile(0.975),
        "draws": draws,
        "seed": seed,
        "family_multiplier": (
            "exponential_mean1_variance1"
        ),
    }


# END_STAGE06_STATISTICS_MODULE

from pathlib import Path

import importlib
import importlib.util
import json
import time
import yaml
import numpy as np
import statsmodels.api as sm


CELL17_STARTED = time.time()

for prior_cell in range(1, 17):
    require_cell_pass(
        prior_cell
    )

statistics_module_path = (
    PROJECT_ROOT
    / "src/statistics.py"
)

statistics_write_status = freeze_visible_module(
    "STATISTICS",
    statistics_module_path,
)

importlib.invalidate_caches()

statistics_spec = importlib.util.spec_from_file_location(
    "stage06_statistics",
    statistics_module_path,
)
if (
    statistics_spec is None
    or statistics_spec.loader is None
):
    raise RuntimeError(
        "Could not load frozen statistics module."
    )
stage06_statistics = (
    importlib.util.module_from_spec(
        statistics_spec
    )
)
statistics_spec.loader.exec_module(
    stage06_statistics
)

fixture_rows = [
    {
        "stratum": "low",
        "d": 0.1,
        "pi": 0.5,
        "family_id": "a",
    },
    {
        "stratum": "low",
        "d": 0.3,
        "pi": 0.5,
        "family_id": "b",
    },
    {
        "stratum": "medium",
        "d": 0.2,
        "pi": 0.5,
        "family_id": "a",
    },
    {
        "stratum": "medium",
        "d": 0.4,
        "pi": 0.5,
        "family_id": "c",
    },
    {
        "stratum": "high",
        "d": 0.0,
        "pi": 0.5,
        "family_id": "d",
    },
    {
        "stratum": "high",
        "d": 0.6,
        "pi": 0.5,
        "family_id": "e",
    },
]

fixture_shares = {
    "low": 1 / 3,
    "medium": 1 / 3,
    "high": 1 / 3,
}

estimated = stage06_statistics.weighted_stratum_effect(
    fixture_rows,
    fixture_shares,
)

expected_effect = (
    0.2 + 0.3 + 0.3
) / 3

if abs(
    estimated["effect"] - expected_effect
) > 1e-12:
    raise RuntimeError(
        "Weighted effect fixture failed."
    )

design = np.asarray([
    [
        1.0 if row["stratum"] == stratum else 0.0
        for stratum in stage06_statistics.STRATA
    ]
    for row in fixture_rows
])
outcome = np.asarray([
    row["d"] for row in fixture_rows
])
weights = np.asarray([
    1.0 / row["pi"] for row in fixture_rows
])
maintained_fit = sm.WLS(
    outcome,
    design,
    weights=weights,
).fit()
expected_stratum_effects = np.asarray([
    estimated["stratum_effects"][stratum]
    for stratum in stage06_statistics.STRATA
])
if not np.allclose(
    maintained_fit.params,
    expected_stratum_effects,
    rtol=0.0,
    atol=1e-12,
):
    raise RuntimeError(
        "Statsmodels WLS fixture disagrees with "
        "the frozen stratum estimator."
    )
maintained_cluster_fit = (
    maintained_fit.get_robustcov_results(
        cov_type="cluster",
        groups=np.asarray([
            row["family_id"] for row in fixture_rows
        ]),
        use_correction=True,
        df_correction=True,
        use_t=True,
    )
)
if not np.isfinite(
    maintained_cluster_fit.cov_params()
).all():
    raise RuntimeError(
        "Statsmodels clustered covariance fixture failed."
    )

manual_cluster = stage06_statistics.family_cluster_standard_error(
    fixture_rows, fixture_shares,
)
maintained_contrast = np.asarray([
    fixture_shares[stratum]
    for stratum in stage06_statistics.STRATA
])
maintained_variance = float(
    maintained_contrast @ maintained_cluster_fit.cov_params() @ maintained_contrast
)
if not np.isclose(
    manual_cluster["variance"], maintained_variance,
    rtol=1e-10, atol=1e-13,
):
    raise RuntimeError(
        "Frozen family-cluster variance disagrees with Statsmodels WLS."
    )

clustered = stage06_statistics.primary_t_inference(
    fixture_rows,
    fixture_shares,
)

if not (
    clustered["standard_error"] > 0
    and clustered["family_count"] == 5
    and 0 < clustered["two_sided_p_value"] <= 1
):
    raise RuntimeError(
        "Clustered inference fixture failed."
    )

holm_result = stage06_statistics.holm_adjustment({
    "a": 0.01,
    "b": 0.03,
    "c": 0.5,
})

if not (
    0 <= holm_result["a"]
    <= holm_result["b"]
    <= holm_result["c"]
    <= 1
):
    raise RuntimeError(
        "Holm monotonicity fixture failed."
    )

bootstrap_result = (
    stage06_statistics.family_multiplier_interval(
        fixture_rows,
        fixture_shares,
        draws=150,
    )
)

if not (
    bootstrap_result["lower"]
    < bootstrap_result["upper"]
):
    raise RuntimeError(
        "Family-bootstrap fixture failed."
    )

qualification_names = [
    "metrics_fixtures",
    "routing_fixtures",
    "real_oracle_routing_preflight",
    "isolation_preflight",
    "isolation_fixtures",
    "integrated_evaluator_fixtures",
]

for report_name in qualification_names:
    report_path = (
        PROJECT_ROOT
        / "reports/qualification"
        / f"{report_name}.json"
    )

    if read_json(
        report_path
    ).get("status") != "PASS":
        raise RuntimeError(
            "Incomplete qualification: "
            + report_name
        )

REGISTERED_ANALYSIS = {
    "schema_version": STATISTICS_VERSION,
    "primary_direction": "L1_minus_L0",
    "design_id": DESIGN,
    "pilot_release_scope": "FRESH_PROBABILITY_SAMPLE",
    "weighted_inference_requires_complete_evidence": True,
    "primary_estimand": (
        "Design-weighted target-stratum paired mean"
    ),
    "inclusion_weights": "1/pi",
    "family_clustering": "family_id",
    "variance_estimator": (
        "family_cluster_sandwich_CR1"
    ),
    "small_sample_correction": (
        "G/(G-1)*(n-1)/(n-k)"
    ),
    "reference_distribution": (
        "Student_t_G_minus_1"
    ),
    "familywise_alpha": 0.05,
    "multiple_comparison": "Holm",
    "primary_endpoint_count": 7,
    "meaningful_effect": 0.10,
    "power_target": 0.80,
    "analysis_seed": 20261105,
    "supplemental_bootstrap": {
        "draws": 10000,
        "family_multiplier": "Exponential(1)",
    },
    "request_match_target": (
        "common_evaluable_pairs"
    ),
    "maintained_statistical_package": (
        "statsmodels"
    ),
    "maintained_package_version": (
        dependency_versions["statsmodels"]
    ),
    "fitting_options": {
        "design": "stratum_indicators_no_intercept",
        "weights": "inverse_inclusion_probability",
        "cluster": "family_id",
        "covariance": "cluster_CR1",
        "reference_distribution": "t_G_minus_1",
    },
    "implementation_path": "src/statistics.py",
    "implementation_sha256": sha256_file(
        statistics_module_path
    ),
}

analysis_config_path = (
    PROJECT_ROOT
    / "config/analysis_registered.yaml"
)

write_immutable(
    analysis_config_path,
    yaml.safe_dump(
        REGISTERED_ANALYSIS,
        sort_keys=False,
    ),
)

FROZEN_DEPENDENCY_PATHS = [
    "config/metrics.yaml",
    "config/analysis_registered.yaml",
    "config/runtime.yaml",
    "config/system_prompt.txt",
    "config/user_prompt.txt",
    "src/metrics.py",
    "src/routing.py",
    "src/worker.py",
    "src/statistics.py",
    "data/tasks.csv",
    "data/audits/cell05_frozen_denominators.jsonl",
    "reports/qualification/canonical_oracle_audit.json",
    "reports/qualification/real_oracle_routing_preflight.json",
    "data/audits/cell15_prompt_token_ledger.jsonl",
]

for prompt in sorted((PROJECT_ROOT / "data/prompts/pilot").rglob("*.txt")):
    FROZEN_DEPENDENCY_PATHS.append(prompt.relative_to(PROJECT_ROOT).as_posix())

l0_documents = sorted((PROJECT_ROOT / "data/inputs/L0").glob("*.json"))
if len(l0_documents) != 100:
    raise RuntimeError("Expected 100 verified L0 operation documents")
FROZEN_DEPENDENCY_PATHS.extend(path.relative_to(PROJECT_ROOT).as_posix()
                               for path in l0_documents)

dependency_hashes = {
    relative_path: sha256_file(
        PROJECT_ROOT / relative_path
    )
    for relative_path
    in FROZEN_DEPENDENCY_PATHS
}

write_json_immutable(
    PROJECT_ROOT
    / "registries/06_frozen_dependency_hashes.json",
    dependency_hashes,
)

FINAL_STAGE06_SUMMARY = {
    "status": "PASS",
    "stage": "06",
    "partition": "pilot",
    "operation_count": 100,
    "task_count": 200,
    "primary_endpoint_count": 7,
    "checkpoint_b_inherited": True,
    "all_evaluator_qualifications_passed": True,
    "model_generation_performed": False,
    "task_ledger_sha256": sha256_file(
        PROJECT_ROOT
        / "data/tasks.csv"
    ),
    "metrics_sha256": dependency_hashes[
        "config/metrics.yaml"
    ],
    "statistics_sha256": dependency_hashes[
        "src/statistics.py"
    ],
    "worker_sha256": dependency_hashes[
        "src/worker.py"
    ],
}

write_json_immutable(
    PROJECT_ROOT
    / "reports/audit/stage_06_final_summary.json",
    FINAL_STAGE06_SUMMARY,
)

cell17_report = save_cell_report(
    17,
    task_count=200,
    primary_endpoints=7,
    registered_analysis_version=STATISTICS_VERSION,
    evaluator_qualification="PASS",
    generation_preflight="PASS",
    model_generation_performed=False,
)

print(
    json.dumps(
        cell17_report,
        indent=2,
    )
)

print("\nNOTEBOOK 06 — CELL 17: PASS")

{
  "task_count": 200,
  "primary_endpoints": 7,
  "registered_analysis_version": "stage06-paired-family-cluster-analysis-v1",
  "evaluator_qualification": "PASS",
  "generation_preflight": "PASS",
  "model_generation_performed": false,
  "status": "PASS",
  "notebook": "06",
  "cell": 17
}

NOTEBOOK 06 — CELL 17: PASS


In [18]:
# NOTEBOOK 06 — CELL 18
# Package and independently verify the Stage 06 Kaggle dataset.

from __future__ import annotations

from pathlib import Path
import hashlib
import json
import os
import time
import zipfile


CELL18_STARTED = time.time()

require_cell_pass(17)

ARCHIVE_NAME = (
    "stage_06_fresh.zip"
)

ARCHIVE_PATH = (
    STAGE_06_DATASET_ROOT
    / ARCHIVE_NAME
)

INTERNAL_MANIFEST_PATH = (
    PROJECT_ROOT
    / "registries/06_manifest.json"
)

EXTERNAL_MANIFEST_PATH = (
    STAGE_06_DATASET_ROOT
    / "stage_06_manifest.json"
)

CHECKSUM_PATH = (
    STAGE_06_DATASET_ROOT
    / "stage_06_checksums.sha256"
)

README_PATH = (
    STAGE_06_DATASET_ROOT
    / "README.md"
)

SUMMARY_PATH = (
    STAGE_06_DATASET_ROOT
    / "stage_summary_for_executed.md"
)

ARCHIVE_ROOT = (
    "stage_06_fresh/"
    "prose_operation_experiment"
)

required_files = [
    "config/runtime.yaml",
    "config/study.yaml",
    "config/system_prompt.txt",
    "config/user_prompt.txt",
    "config/stage06_environment.json",
    "config/oracle_record_contracts.json",
    "config/metrics.yaml",
    "data/audits/cell04_pair_reconstruction.jsonl",
    "config/analysis_registered.yaml",
    "src/metrics.py",
    "src/routing.py",
    "src/worker.py",
    "src/statistics.py",
    "data/tasks.csv",
    "data/inventory/final_selected_operations.json",
    "data/audits/cell04_pair_reconstruction.jsonl",
    "data/audits/cell05_frozen_denominators.jsonl",
    "data/audits/cell14_rendered_prompt_ledger.jsonl",
    "data/audits/cell15_prompt_token_ledger.jsonl",
    "registries/06_frozen_dependency_hashes.json",
    "reports/audit/stage_06_final_summary.json",
    "reports/audit/"
    "upstream_stage_00_04_05_verification.json",
]

for cell_number in range(1, 18):
    required_files.append(
        "reports/audit/"
        f"cell{cell_number:02d}_stage06.json"
    )

for stage in (
    "00",
    "04",
    "05",
):
    required_files.append(
        "reports/audit/"
        f"upstream_stage_{stage}_verified.json"
    )

for qualification in (
    "metrics_fixtures",
    "routing_fixtures",
    "real_oracle_routing_preflight",
    "canonical_oracle_audit",
    "isolation_preflight",
    "isolation_fixtures",
    "integrated_evaluator_fixtures",
):
    required_files.append(
        "reports/qualification/"
        f"{qualification}.json"
    )

prompt_directory = (
    PROJECT_ROOT
    / "data/prompts/pilot"
)

prompt_files = sorted(
    prompt_directory.rglob(
        "*.txt"
    )
)

if len(prompt_files) != 200:
    raise RuntimeError(
        "Expected exactly 200 prompt files."
    )

required_files.extend(path.relative_to(PROJECT_ROOT).as_posix()
                      for path in l0_documents)

for prompt_path in prompt_files:
    required_files.append(
        prompt_path.relative_to(
            PROJECT_ROOT
        ).as_posix()
    )

required_files = sorted(
    set(required_files)
)

for relative_path in required_files:
    full_path = (
        PROJECT_ROOT
        / relative_path
    )

    if not full_path.is_file():
        raise RuntimeError(
            "Required Stage 06 artifact missing: "
            + relative_path
        )

UPSTREAM_MANIFEST_HASHES = {
    stage: sha256_bytes(
        STAGE_ARCHIVES[stage].read(
            f"registries/{stage}_manifest.json"
        )
    )
    for stage in (
        "00",
        "04",
        "05",
    )
}

CONFIG_HASH = sha256_bytes(
    canonical_json_bytes({
        relative_path: sha256_file(
            PROJECT_ROOT / relative_path
        )
        for relative_path in required_files
        if relative_path.startswith("config/")
    })
)

artifact_records = []

for relative_path in required_files:
    full_path = (
        PROJECT_ROOT
        / relative_path
    )

    artifact_records.append({
        "relative_path": relative_path,
        "bytes": full_path.stat().st_size,
        "sha256": sha256_file(
            full_path
        ),
    })

internal_manifest = {
    "stage": "06",
    "design_id": DESIGN,
    "status": "PASS",
    "partition": "pilot",
    "manifest_version": (
        "stage06-lean-evaluator-tasks-v4"
    ),
    "packaging_mode": (
        "LEAN_NONCUMULATIVE_STAGE_PACKAGE"
    ),
    "checkpoint_b_inherited": True,
    "model_generation_performed": False,
    "operation_count": 100,
    "task_count": 200,
    "primary_endpoint_count": 7,
    "evaluator_correction": "SEMANTIC_REQUEST_CONSTRAINT_ROUTING_AND_FAILURE_ATTRIBUTION",
    "artifact_count": len(
        artifact_records
    ),
    "artifact_bytes": sum(
        row["bytes"]
        for row in artifact_records
    ),
    "configuration_hash": CONFIG_HASH,
    "artifacts": artifact_records,
    "evaluation_worker_version": worker.WORKER_VERSION,
    "verified_operation_document_count": len(l0_documents),
    "human_review_complete": True,
    "upstream_manifest_sha256": (
        UPSTREAM_MANIFEST_HASHES
    ),
}

register_artifact_provenance(internal_manifest)

internal_manifest_bytes = (
    canonical_json_bytes(
        internal_manifest
    )
)

write_immutable(
    INTERNAL_MANIFEST_PATH,
    internal_manifest_bytes,
)

archive_members = (
    required_files
    + ["registries/06_manifest.json"]
)

temporary_archive = (
    ARCHIVE_PATH.with_suffix(
        ".zip.tmp"
    )
)

if not ARCHIVE_PATH.exists():
    with zipfile.ZipFile(
        temporary_archive,
        mode="w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=7,
        allowZip64=True,
    ) as archive:
        for relative_path in archive_members:
            full_path = (
                PROJECT_ROOT
                / relative_path
            )

            member_name = (
                ARCHIVE_ROOT
                + "/"
                + relative_path
            )

            info = zipfile.ZipInfo(
                member_name,
                date_time=(
                    2026,
                    11,
                    11,
                    0,
                    0,
                    0,
                ),
            )

            info.compress_type = (
                zipfile.ZIP_DEFLATED
            )

            archive.writestr(
                info,
                full_path.read_bytes(),
                compress_type=zipfile.ZIP_DEFLATED,
                compresslevel=7,
            )

    os.replace(
        temporary_archive,
        ARCHIVE_PATH,
    )

with zipfile.ZipFile(
    ARCHIVE_PATH
) as archive:
    actual_members = set(
        archive.namelist()
    )

    expected_members = {
        ARCHIVE_ROOT
        + "/"
        + relative_path
        for relative_path in archive_members
    }

    if actual_members != expected_members:
        raise RuntimeError(
            "Stage 06 ZIP member list mismatch."
        )

    if archive.testzip() is not None:
        raise RuntimeError(
            "Stage 06 ZIP CRC verification failed."
        )

    for artifact in artifact_records:
        member_name = (
            ARCHIVE_ROOT
            + "/"
            + artifact["relative_path"]
        )

        archived_bytes = archive.read(
            member_name
        )

        if (
            sha256_bytes(archived_bytes)
            != artifact["sha256"]
        ):
            raise RuntimeError(
                "Stage 06 ZIP artifact mismatch: "
                + artifact["relative_path"]
            )

    archived_manifest = archive.read(
        ARCHIVE_ROOT
        + "/registries/06_manifest.json"
    )

    if (
        archived_manifest
        != internal_manifest_bytes
    ):
        raise RuntimeError(
            "Archived internal manifest differs."
        )

external_manifest = {
    "stage": "06",
    "design_id": DESIGN,
    "status": "PASS",
    "partition": "pilot",
    "archive_name": ARCHIVE_NAME,
    "archive_sha256": sha256_file(
        ARCHIVE_PATH
    ),
    "archive_bytes": (
        ARCHIVE_PATH.stat().st_size
    ),
    "archive": {
        "filename": ARCHIVE_NAME,
        "sha256": sha256_file(ARCHIVE_PATH),
        "bytes": ARCHIVE_PATH.stat().st_size,
        "member_count": len(archive_members),
        "project_prefix": ARCHIVE_ROOT,
    },
    "internal_manifest_sha256": (
        sha256_bytes(
            internal_manifest_bytes
        )
    ),
    "internal_manifest": {
        "relative_path": (
            "registries/06_manifest.json"
        ),
        "sha256": sha256_bytes(
            internal_manifest_bytes
        ),
        "artifact_count": len(artifact_records),
    },
    "operation_count": 100,
    "task_count": 200,
    "primary_endpoint_count": 7,
    "checkpoint_b_inherited": True,
    "evaluator_qualification": "PASS",
    "evaluator_correction": "SEMANTIC_REQUEST_CONSTRAINT_ROUTING_AND_FAILURE_ATTRIBUTION",
    "existing_stage07_replay_requires_exact_task_and_prompt_hash_match": True,
    "model_generation_performed": False,
    "may_begin_notebook_07": True,
    "packaging_mode": (
        "LEAN_NONCUMULATIVE_STAGE_PACKAGE"
    ),
}

write_json_immutable(
    EXTERNAL_MANIFEST_PATH,
    external_manifest,
)

checksum_text = (
    external_manifest[
        "archive_sha256"
    ]
    + "  "
    + ARCHIVE_NAME
    + "\n"
    + external_manifest[
        "internal_manifest_sha256"
    ]
    + "  06_manifest.json\n"
)

write_immutable(
    CHECKSUM_PATH,
    checksum_text,
)

readme_text = """
# Stage 06 fresh pilot — Corrected Pilot Evaluator and Generation Tasks

Status: PASS

This dataset contains:
- Seven frozen primary scoring metrics.
- Qualified offline routing and isolated worker modules.
- Semantic request validation; frozen examples are not treated as exhaustive values.
- Evaluator-unavailable outcomes are separated from generated-test failures.
- Evaluator qualification evidence.
- Frozen analysis implementation.
- 200 immutable L0/L1 prompts.
- Exactly 200 generation tasks covering 100 pilot operations.
- Internal manifest and immutable dependency hashes.

Generation has NOT been performed.

Notebook 07 must attach Stage 00 and Stage 06.
Notebook 08 must separately attach Stage 05 oracles.

Reference:
- PLAN.md, Notebook 06, Cells 1–18.
- Project Pipeline.md, Section 11.

The archive is lean and does not contain the upstream corpus
or duplicate frozen oracle packages.
""".strip() + "\n"

write_immutable(
    README_PATH,
    readme_text,
)

summary_text = """
# Notebook 06 Corrected — PASS

- Pilot operations: 100
- Rendered prompts: 200
- Generation tasks: 200
- Frozen primary metrics: 7
- Evaluator qualification: PASS
- Checkpoint B: inherited from Stage 05
- Model generation: not performed
- Notebook 07 authorized: yes

Stage 06 package and all registered artifact hashes
were independently reconciled.
""".strip() + "\n"

write_immutable(
    SUMMARY_PATH,
    summary_text,
)

final_report = {
    "status": "PASS",
    "stage": "06",
    "design_id": DESIGN,
    "partition": "pilot",
    "operation_count": 100,
    "prompt_count": 200,
    "task_count": 200,
    "archive_path": str(
        ARCHIVE_PATH
    ),
    "archive_sha256": (
        external_manifest[
            "archive_sha256"
        ]
    ),
    "internal_manifest_sha256": (
        external_manifest[
            "internal_manifest_sha256"
        ]
    ),
    "artifact_count": len(
        artifact_records
    ),
    "model_generation_performed": False,
    "may_begin_notebook_07": True,
}

print(
    json.dumps(
        final_report,
        indent=2,
    )
)

print("\nNOTEBOOK 06 — CELL 18: PASS")
print(
    "STAGE 06: FROZEN — "
    "100 operations, 200 prompts, 200 tasks."
)
print(
    "NEXT: Upload the Stage 06 dataset "
    "and proceed to Notebook 07."
)

{
  "status": "PASS",
  "stage": "06",
  "design_id": "openapi-fresh-pilot-v1",
  "partition": "pilot",
  "operation_count": 100,
  "prompt_count": 200,
  "task_count": 200,
  "archive_path": "/kaggle/working/stage_06_fresh_dataset/stage_06_fresh.zip",
  "archive_sha256": "c52507e6d43d48695d5ac53d341d05a026b7e9a57f35fdd75465b903e2b87d81",
  "internal_manifest_sha256": "7daa321303458a29bbbc1e8e5c077a46257cccf6617543071b9232a782725b52",
  "artifact_count": 348,
  "model_generation_performed": false,
  "may_begin_notebook_07": true
}

NOTEBOOK 06 — CELL 18: PASS
STAGE 06: FROZEN — 100 operations, 200 prompts, 200 tasks.
NEXT: Upload the Stage 06 dataset and proceed to Notebook 07.
